# Myeloid and monocyte analysis

## Repository navigation

This notebook is a downstream analytical workflow record for cross-disease myeloid and monocyte analyses. It starts from a processed myeloid/monocyte AnnData object and records the analysis logic used for metadata harmonization, myeloid visualization, cytokine-storm and functional-module scoring, composition analysis, DEG/enrichment analysis, COVID-19 and SLE monocyte analyses, and CAR-T-associated monocyte analyses.

# 1. Load processed myeloid AnnData object

Environment setup and loading of the processed myeloid/monocyte object. This notebook does not perform raw-data preprocessing, alignment, counting, or initial matrix generation.


In [1]:
import os

# Set environment variables to limit the number of threads used by various libraries
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"
os.environ["VECLIB_MAXIMUM_THREADS"] = "16"
os.environ["NUMEXPR_NUM_THREADS"] = "16"

# Documented private input and output path variables
# The processed AnnData object is not distributed
# MYELOID_H5AD and MYELOID_OUTPUT_DIR record the expected environment-variable
# names used in the original downstream workflow
analysis_start_dir = globals().get("analysis_start_dir", os.getcwd())

myeloid_h5ad_setting = os.environ.get("MYELOID_H5AD", "adata_myeloid.h5ad")
myeloid_h5ad_path = (
    myeloid_h5ad_setting
    if os.path.isabs(myeloid_h5ad_setting)
    else os.path.join(analysis_start_dir, myeloid_h5ad_setting)
)

myeloid_output_setting = os.environ.get(
    "MYELOID_OUTPUT_DIR", os.path.join("outputs", "myeloid")
)
myeloid_output_dir = (
    myeloid_output_setting
    if os.path.isabs(myeloid_output_setting)
    else os.path.join(analysis_start_dir, myeloid_output_setting)
)
os.makedirs(myeloid_output_dir, exist_ok=True)


In [2]:
# Main Dependency
# Python version: 3.11.13

# Load Python packages
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from scipy import stats
import gseapy as gp  # for functional enrichment analysis

In [3]:
# Global settings

# figure parameters
sc.settings.set_figure_params(frameon=False, dpi=300)
sc.set_figure_params(dpi_save=300)

# Suppress scanpy warnings
sc.settings.verbosity = 2  

In [ ]:
# Load and validate the processed myeloid-specific AnnData object, then
# direct relative outputs to the configured analysis-output directory.
adata_m = sc.read_h5ad(myeloid_h5ad_path)

In [12]:
if adata_m.n_obs == 0 or adata_m.n_vars == 0:
    raise ValueError('The myeloid AnnData object must contain cells and genes.')
if not adata_m.obs_names.is_unique:
    raise ValueError('adata_m.obs_names must be unique.')
if not adata_m.var_names.is_unique:
    raise ValueError('adata_m.var_names must be unique.')

required_myeloid_obs_columns = [
    'sample_id', 'disease', 'stage','severity',
    'myeloid_anno'
]
missing_myeloid_obs_columns = [
    column for column in required_myeloid_obs_columns
    if column not in adata_m.obs.columns
]
if missing_myeloid_obs_columns:
    raise ValueError(
        f"Missing required myeloid metadata columns: {missing_myeloid_obs_columns}"
    )

if 'X_umap' not in adata_m.obsm:
    raise ValueError("Missing required embedding: adata_m.obsm['X_umap']")
if adata_m.obsm['X_umap'].shape[0] != adata_m.n_obs:
    raise ValueError("adata_m.obsm['X_umap'] does not match the number of cells.")
if adata_m.obsm['X_umap'].ndim != 2 or adata_m.obsm['X_umap'].shape[1] < 2:
    raise ValueError("adata_m.obsm['X_umap'] must contain at least two dimensions.")

# Several preserved DEG and scoring blocks explicitly use the stored raw gene space.
if adata_m.raw is None:
    raise ValueError('adata_m.raw is required by downstream DEG and scoring sections.')
if adata_m.raw.n_vars == 0:
    raise ValueError('adata_m.raw must contain genes.')
if adata_m.raw.X.shape[0] != adata_m.n_obs:
    raise ValueError('adata_m.raw does not match the number of cells in adata_m.')
if not adata_m.raw.var_names.is_unique:
    raise ValueError('adata_m.raw.var_names must be unique.')

expected_myeloid_diseases = {'CAR-T_CRS', 'COVID19', 'SLE', 'HC'}
missing_myeloid_diseases = expected_myeloid_diseases - set(
    adata_m.obs['disease'].dropna().astype(str)
)
if missing_myeloid_diseases:
    raise ValueError(
        f"Missing expected myeloid disease categories: {sorted(missing_myeloid_diseases)}"
    )

expected_myeloid_stages = {
    'CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con',
    'COVID19_pro', 'COVID19_con', 'SLE', 'HC'
}
missing_myeloid_stages = expected_myeloid_stages - set(
    adata_m.obs['stage'].dropna().astype(str)
)
if missing_myeloid_stages:
    raise ValueError(
        f"Missing expected myeloid stage categories: {sorted(missing_myeloid_stages)}"
    )

expected_myeloid_severities = {'Moderate', 'Severe', 'No_CRS', 'HC'}
missing_myeloid_severities = expected_myeloid_severities - set(
    adata_m.obs['severity'].dropna().astype(str)
)
if missing_myeloid_severities:
    raise ValueError(
        f"Missing expected myeloid severity categories: {sorted(missing_myeloid_severities)}"
    )

if adata_m.obs['sample_id'].isna().any():
    raise ValueError('adata_m.obs contains missing sample_id values.')
if adata_m.obs['myeloid_anno'].isna().any():
    raise ValueError('adata_m.obs contains missing myeloid_anno values.')

os.chdir(myeloid_output_dir)
sc.settings.figdir = '.'
print(f"Loaded myeloid object: {adata_m.n_obs:,} cells × {adata_m.n_vars:,} genes")

Loaded myeloid object: 351,474 cells × 16,977 genes


# 2. Myeloid/monocyte annotation overview

Metadata harmonization, annotation inspection, and downstream filtering operate on the processed analysis object.

In [14]:
print(adata_m.obs.columns)

Index(['sample_id', 'batch', 'n_genes', 'n_genes_by_counts', 'total_counts',
       'total_counts_mt', 'pct_counts_mt', 'disease', 'severity', 'stage',
       'myeloid_anno'],
      dtype='object')


In [15]:
# Check specific contents in column
for s in adata_m.obs['disease'].unique():
    print(s)

CAR-T_CRS
HC
SLE
COVID19


In [16]:
# Create one 'group' column that combines 'stage' and 'severity'
adata_m.obs['group'] = adata_m.obs['stage'].astype(str) + '_' + adata_m.obs['severity'].astype(str)

# 3. Myeloid annotation UMAP

In [ ]:
# Major UMAP (Fig. 2B)

plt.rcParams['figure.dpi'] = 300

myeloid_palette = {
    'Mono_CD14_IL1B': '#457B9D',
    'Mono_CD14_S100A8': '#F4A261',
    'Mono_CD14_IFI44': '#A6CEE3',
    'Mono_CD14_CD16': '#2A9D8F',
    'Mono_CD16_LST1': '#8E44AD',
    'cDC_CD1C': '#E9C46A'
}

# Set the layout
fig, ax = plt.subplots(figsize=(6, 6))

# plot UMAP
sc.pl.umap(
    adata_m,
    color = 'myeloid_anno',
    palette = myeloid_palette,
    frameon = False,
    size = 3,
    legend_loc='none',
    ax=ax,
    show=False
)

# save pdf
plt.savefig('Myeloid_subclusters_umap.pdf', bbox_inches='tight')

In [ ]:
# Separated UMAPs of diseases (Fig. S2A)

# Get unique diseases
diseases = adata_m.obs['disease'].unique()

for disease in diseases:
    # Create a new figure for each disease
    fig, ax = plt.subplots(figsize=(4, 6))
    
    # Plot background (all cells)
    sc.pl.umap(
        adata_m,
        ax=ax,
        size=1,  # Smaller size for background points
        show=False,
        title=disease,
        frameon=False
    )
    
    # Overlay the specific disease data
    subset = adata_m[adata_m.obs['disease'] == disease]
    sc.pl.umap(
        subset,
        color='myeloid_anno',
        palette=myeloid_palette,
        size=3,
        ax=ax,
        legend_loc='none',
        frameon=False,
        show=False,
        title=disease
    )
    
    # Save separately
    # Sanitize filename
    disease_name = str(disease).replace(' ', '_').replace('/', '_')
    plt.savefig(f'Myeloid_subclusters_umap_{disease_name}.pdf', bbox_inches='tight')

# 4. Cross-disease monocyte subcluster Ro/e
---
Fig. 2E

In [19]:
# Only retain concrete monocyte subclusters
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']

adata_mo = adata_m[~adata_m.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

# only keep groups of interest (active diseases)
groups_to_keep = ['CAR-T_CRS_pro_Severe', 'COVID19_pro_Severe', 'SLE_Severe']
adata_mo = adata_mo[adata_mo.obs['group'].isin(groups_to_keep)].copy()

In [ ]:
# Ro/e analysis

# Observed counts: rows=subclusters, cols=groups
obs_counts = pd.crosstab(adata_mo.obs['myeloid_anno'], adata_mo.obs['group'])

# Expected counts under independence
row_totals = obs_counts.sum(axis=1).values[:, None]
col_totals = obs_counts.sum(axis=0).values[None, :]
grand_total = obs_counts.values.sum()

exp_counts = pd.DataFrame(
    (row_totals @ col_totals) / grand_total,
    index=obs_counts.index,
    columns=obs_counts.columns
)

# Ro/e
eps = 1e-9
roe = (obs_counts + eps) / (exp_counts + eps)

# Heatmap visualization using raw Ro/e
plt.figure(figsize=(12, 8))
sns.heatmap(
    roe,
    cmap="RdBu_r",
    center=1,
    linewidths=0.2,
    linecolor="white",
    annot=True, # show values
    fmt=".2f",
    cbar_kws={"label": "Ro/e"}
)

plt.title("Ro/e enrichment of myeloid subclusters across groups")
plt.xlabel("Group")
plt.ylabel("Myeloid subcluster")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("myeloid_subcluster_RoE_heatmap.pdf", dpi=300, bbox_inches="tight")
plt.show()

# 5. Myeloid CS scoring

## Calculation

In [ ]:
# Myeloid-lineage curated CS genes 
cs_genes = [
    # Classical cytokines (both pro-inflammatory and regulatory cytokines)
    'IL1A', 'IL1B', 'IL6', 'TNF', 'IFNG', 'IL10', 'IL18', 'IFNB1', 'CSF2', 'CSF3',
    # Key receptors for key cytokines
    'IL1R1', 'IL6R', 'IL6ST', 'TNFRSF1A', 'TNFRSF1B','CSF2RA', 'CSF3R', 'IFNAR1', 'IFNAR2',
    # Key chemokines and their receptors
    'CCL2', 'CCR2', 'CCL5', 'CCR5', 'CXCR1', 'CXCR2', 'CXCL1',
    # Cytokine-induced/related signaling and regulatory molecules
    'NFKB1', 'NFKBIA', 'NFKBIZ', 'STAT1', 'STAT3', 'SOCS3', 'IRF4', 'JUN', 'FOS', 'IFI27',
    'TNFAIP6', 'CEBPB', 'PTGS2', 
    # Innate immune hyperactivation markers
    'S100A8', 'S100A9', 'S100A12', 'NLRP3', 'CASP1', 'GSDMD', 'TLR2', 'ISG15', 'MX1'
]

In [23]:
sc.tl.score_genes(
    adata_m, 
    gene_list=cs_genes, 
    score_name='CS_score', 
    use_raw=False,
    ctrl_size=len(cs_genes)
)

computing score 'CS_score'
    finished (0:00:10)


## CS gene expression across subclusters and groups
---
Fig. S2D, Table S3b

In [ ]:
# Subset the adata_m to CAR-T_CRS_pro_Severe, COVID19_pro_Severe, and SLE_Severe groups
adata_m_s = adata_m[adata_m.obs['group'].isin(['CAR-T_CRS_pro_Severe', 'COVID19_pro_Severe', 'SLE_Severe']), :].copy()

# Drop subclusters that are not of interest (sample-biased)
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16']
adata_m_s = adata_m_s[~adata_m_s.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

# Calculate mean expression of CS genes for each myeloid subcluster in each group
# Generate 3 separated dataframes for the 3 groups
# In each dataframe, rows are myeloid subclusters, columns are CS genes, and values are mean expression
# Build expression table with metadata 
expr_df = adata_m_s.to_df()[cs_genes].copy()
expr_df['myeloid_anno'] = adata_m_s.obs['myeloid_anno'].values
expr_df['group'] = adata_m_s.obs['group'].values

# Mean expression by (group, myeloid_anno)
mean_expr_all = expr_df.groupby(['group', 'myeloid_anno'])[cs_genes].mean()

# Generate 3 separated dataframes
# rows: CS genes, columns: myeloid subclusters
mean_expr_cart = mean_expr_all.xs('CAR-T_CRS_pro_Severe', level='group').T
mean_expr_covid19 = mean_expr_all.xs('COVID19_pro_Severe', level='group').T
mean_expr_sle = mean_expr_all.xs('SLE_Severe', level='group').T

# Save into one xlsx file with separate sheets (metadata for Table S3b and Fig. S2D)
with pd.ExcelWriter('CS_genes_mean_expression_by_group.xlsx') as writer:
    mean_expr_cart.to_excel(writer, sheet_name='CAR-T_CRS_pro_Severe')
    mean_expr_covid19.to_excel(writer, sheet_name='COVID19_pro_Severe')
    mean_expr_sle.to_excel(writer, sheet_name='SLE_Severe')

## Myeloid subcluster abundance and CS scores across groups
---
Fig. 2C

In [26]:
# Subset adata_m to remove No CRS and nan samples
adata_sub = adata_m[~adata_m.obs['severity'].isin(['No_CRS', 'nan']), :].copy()

# Subset to only groups of interest (progression-stage samples)
groups_of_interest = ['CAR-T_CRS_pro', 'COVID19_pro', 'SLE', 'HC']
adata_sub = adata_sub[adata_sub.obs['stage'].isin(groups_of_interest)].copy()

In [ ]:
# Filter subclusters that are not interested
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16']
adata_sub = adata_sub[~adata_sub.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

# Calculate the fraction of each myeloid subcluster in each group
count_df = adata_sub.obs.groupby(['stage', 'myeloid_anno']).size().unstack(fill_value=0)
fraction_df = count_df.div(count_df.sum(axis=1), axis=0).T

# Calculate raw CS score means (no Z-score)
cs_score_mean = adata_sub.obs.groupby(['myeloid_anno', 'stage'])['CS_score'].mean().unstack()

# Align clusters present in both tables
common_clusters = fraction_df.index.intersection(cs_score_mean.index)
fraction_df = fraction_df.loc[common_clusters]
cs_score_mean = cs_score_mean.loc[common_clusters]

# Prepare data for plotting (Long format)
cs_score_long = cs_score_mean.reset_index().melt(
    id_vars='myeloid_anno', var_name='stage', value_name='CS_score'
)
fraction_long = fraction_df.reset_index().melt(
    id_vars='myeloid_anno', var_name='stage', value_name='fraction'
)

# Merge
plot_data = pd.merge(cs_score_long, fraction_long, on=['myeloid_anno', 'stage'])

# Filter groups and order
group_order = ['CAR-T_CRS_pro', 'COVID19_pro', 'SLE', 'HC']
plot_data = plot_data[plot_data['stage'].isin(group_order)]
plot_data['stage'] = pd.Categorical(
    plot_data['stage'], categories=group_order, ordered=True
)
plot_data = plot_data.sort_values('stage')

# Plotting
plt.figure(figsize=(8, 6))
sc_plot = plt.scatter(
    x=plot_data['stage'],
    y=plot_data['myeloid_anno'],
    s=plot_data['fraction'] * 1000,
    c=plot_data['CS_score'],
    cmap='RdBu_r'
)

plt.xticks(rotation=90)
plt.colorbar(sc_plot, label='Raw CS Score')
plt.xlabel('Stage')
plt.ylabel('Myeloid Subcluster')
plt.title('Raw CS Score and Subcluster Fraction')

# Legend for size
sizes = [0.05, 0.1, 0.2, 0.4]
legend_elements = [plt.scatter([], [], s=s*1000, c='gray', label=f'{s*100:.0f}%') for s in sizes]
plt.legend(handles=legend_elements, title='Fraction', bbox_to_anchor=(1.2, 1), loc='upper left')

plt.savefig('Myeloid_CS_score_dotplot_rawCS.pdf', bbox_inches='tight', dpi=300)
plt.tight_layout()
plt.show()

# 6. IL1B versus S100A8 versus CD16 monocytes
---
Fig. 2H, Fig. S2F, Table S3c

In [29]:
# Subset the adata_m to remove No CRS and SLE_None and HC samples
adata_sub = adata_m[~adata_m.obs['severity'].isin(['No_CRS', 'None', 'HC'])].copy()

# Only retain subclusters of interest
adata_sub = adata_sub[adata_sub.obs['myeloid_anno'].isin(['Mono_CD14_IL1B', 'Mono_CD14_S100A8', 'Mono_CD16_LST1'])].copy()

In [30]:
gene_dict = {
    'Emergency_myelopoiesis': ['MPO', 'LTF', 'VCAN', 'RETN', 'LCN2', 'S100A12', 'MMP8', 'OLR1'],
    'Cytokine_chemokine': ['IL1B', 'IL1A', 'IL6', 'TNF', 'IL18', 'CCL2', 'CCL3', 'CCL4'],
    'Antigen_presentation': ['HLA-DRA', 'HLA-DRB1', 'CD74', 'CIITA']
}

for module, genes in gene_dict.items():
    valid_genes = [gene for gene in genes if gene in adata_sub.raw.var_names]
    if len(valid_genes) > 0:
        sc.tl.score_genes(adata_sub, gene_list=valid_genes, score_name=module, use_raw=False)
    else:
        print(f"Warning: No valid genes found for module {module}")

computing score 'Emergency_myelopoiesis'
    finished (0:00:07)
computing score 'Cytokine_chemokine'
    finished (0:00:07)
computing score 'Antigen_presentation'
    finished (0:00:07)


In [ ]:
# Generate separated dotplots for each module (Fig. S2F)
dp = sc.pl.dotplot(
    adata_sub, 
    var_names = gene_dict,
    groupby = 'myeloid_anno',
    use_raw = False,
    cmap='OrRd',
    standard_scale='var',
    swap_axes=True,
    dot_min=0,
    figsize=(5,15),
    return_fig=True
)

dp.make_figure()

scatter_artist = dp.ax_dict['mainplot_ax'].collections[0]

original_sizes = scatter_artist.get_sizes()

min_size = 20
new_size = original_sizes.copy()
new_size[new_size > 0] += min_size

scatter_artist.set_sizes(new_size)

scatter_artist.set_linewidths(0)

dp.fig.savefig('IL1B_S100A8_CD16_functional_module_genes.pdf', bbox_inches='tight', dpi=300)

plt.show()

In [32]:
# Output sample-level means (Table S3c, Fig. 2H)

scores_to_output = ['Emergency_myelopoiesis', 'Cytokine_chemokine', 'Antigen_presentation']

# Calculate sample means for each score and subcluster
sample_means = (
    adata_sub.obs
    .groupby(['sample_id', 'myeloid_anno'], observed=True)[scores_to_output]
    .mean()
    .reset_index()
)

# Target subclusters for export
target_subclusters = ['Mono_CD14_IL1B', 'Mono_CD14_S100A8', 'Mono_CD16_LST1']

# Export wide-format sample means to Excel, one sheet per score
with pd.ExcelWriter("IL1B_vs_S100A8_sample_means.xlsx", engine="openpyxl") as writer:
    for score in scores_to_output:
        wide_df = (
            sample_means.pivot(index='sample_id', columns='myeloid_anno', values=score)
            .reindex(columns=target_subclusters)
            .reset_index()
        )
        wide_df.to_excel(writer, sheet_name=score, index=False)

# 7. Monocyte DEG analyses

## Global monocyte DEGs

### Monocyte DEGs calculation

In [ ]:
# Build a temporary object for DEG calculation
adata_m_deg = sc.AnnData(
    X = adata_m.raw.X.copy(),
    obs = adata_m.obs.copy(),
    var = adata_m.raw.var.copy()
)

sc.pp.normalize_total(adata_m_deg, target_sum=1e4)
sc.pp.log1p(adata_m_deg)

In [ ]:
# Calculate the DEGs of monocytes against the rest subclusters
sub_of_interest = ['Mono_CD14_IL1B', 'Mono_CD14_S100A8', 'Mono_CD16_LST1']

# Loop through each subcluster of interest
for subcluster in sub_of_interest:
    print(f"Analyzing DEGs for {subcluster} against all other myeloid cells...")
    
    # Create a subset of the data
    adata_subset = adata_m_deg[adata_m_deg.obs['myeloid_anno'].isin([subcluster] + 
                                                                  [x for x in adata_m_deg.obs['myeloid_anno'].unique() if x != subcluster])].copy()
    
    # Add a new column to distinguish the focal subcluster from others
    adata_subset.obs['comparison_group'] = adata_subset.obs['myeloid_anno'].apply(
        lambda x: subcluster if x == subcluster else 'Other'
    )
    
    # Perform DEG analysis
    sc.tl.rank_genes_groups(
        adata_subset,
        groupby='comparison_group',
        groups=[subcluster],
        reference='Other',
        method='wilcoxon',
        use_raw=False
    )
    
    # Get results
    result_df = sc.get.rank_genes_groups_df(adata_subset, group=subcluster)
    
    # Filter significant genes
    significant_genes = result_df[result_df['pvals_adj'] < 0.05]
    
    # Save results to a CSV file (metadata for Table S3a)
    filename = f"DEGs_{subcluster.replace('+', '_').replace('-', '_')}_vs_Other.csv"
    significant_genes.to_csv(filename, index=False)

### Monocyte DEG volcano plot
---
Fig. 2F

In [ ]:
# DEG plot for key DEGs of IL1B, S100A8, and CD16_LST1 monocytes
# Input files:
# - DEGs_Mono_CD14_IL1B_vs_Other.csv
# - DEGs_Mono_CD14_S100A8_vs_Other.csv
# - DEGs_Mono_CD16_LST1_vs_Other.csv

from adjustText import adjust_text

# ---Read data---
df1 = pd.read_csv('DEGs_Mono_CD14_IL1B_vs_Other.csv').assign(group='Mono_IL1B')
df2 = pd.read_csv('DEGs_Mono_CD14_S100A8_vs_Other.csv').assign(group='Mono_S100A8')
df3 = pd.read_csv('DEGs_Mono_CD16_LST1_vs_Other.csv').assign(group='Mono_CD16_LST1')

df = pd.concat([df1, df2, df3], ignore_index=True)

# Filter for upregulated DEGs with logfoldchanges > 0.5
df = df[df['logfoldchanges'] > 0.5].copy()

# ---label---
label = {
    'Mono_IL1B': [
        'IL1B', 'FOSB', 'CCL3', 'TNFAIP3', 'NFKBIA', 'NLRP3', 'CCL4', 'CXCL2', 'IL1A', 'TNF', 'CCR1', 'IL6'
    ],
    'Mono_S100A8': [
        'S100A12', 'VCAN', 'S100A8', 'CD14', 'RETN', 'ITGAM', 'IL1R2', 'ELANE', 'LCN2'
    ],
    'Mono_CD16_LST1': [
        'FCGR3A', 'CSF1R', 'CX3CR1', 'IFITM2', 'IFITM1', 'IFITM3', 'ISG15', 'HLA-DPA1', 'HLA-DPB1', 'TNF'
    ]
}

# ---custom color palette---
group_palette = {
    'Mono_IL1B': '#447296',
    'Mono_S100A8': '#DB9B6D',
    'Mono_CD16_LST1': '#4F3D80'
}

# ---point size---
df['point_size'] = np.log1p(np.abs(df['scores']))

# ---assign labels---
df['label'] = df.apply(
    lambda r: r['names'] if r['names'] in label[r['group']] else None,
    axis=1
)

# ---group order---
group_order = ['Mono_IL1B', 'Mono_S100A8', 'Mono_CD16_LST1']
x_map = {g: i for i, g in enumerate(group_order)}
df['x'] = df['group'].map(x_map)

# ---grey background ranges---
shades = (
    df.groupby('group')['logfoldchanges']
    .agg(['min', 'max'])
    .reset_index()
)
shades['x'] = shades['group'].map(x_map)

# ---colors---
df['color'] = np.where(
    df['label'].notna(),
    df['group'].map(group_palette),
    "#b3b1b1ff"
)

# ---jitter---
rng = np.random.default_rng(seed=1)
df['x_jitter'] = df['x'] + rng.uniform(-0.3, 0.3, size=len(df))

# Plot unlabeled first, labeled last
df['is_labeled'] = df['label'].notna()
df = df.sort_values('is_labeled', ascending=True)

# ---plot---
fig, ax = plt.subplots(figsize=(8, 6))

for _, r in shades.iterrows():
    if r['max'] > 0.5:
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, 0.5),
                width=0.7,
                height=r['max'] - 0.5 + 0.5,
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )

ax.scatter(
    df['x_jitter'],
    df['logfoldchanges'],
    s=df['point_size'] * 50,
    c=df['color'],
    alpha=0.8,
    zorder=1
)

texts = []
for _, r in df.dropna(subset=['label']).iterrows():
    texts.append(
        ax.text(
            r['x_jitter'],
            r['logfoldchanges'],
            r['label'],
            fontsize=8,
            color='black'
        )
    )

adjust_text(
    texts,
    ax=ax,
    only_move={'points': 'y', 'texts': 'y'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8),
    lim=200
)

# ---axes---
ax.set_xlim(-0.6, len(group_order) - 0.4)

y_max = df['logfoldchanges'].max() + 1
ax.set_ylim(0, y_max)

ax.set_xticks(list(x_map.values()))
ax.set_xticklabels(list(x_map.keys()))
ax.set_xlabel('')
ax.set_ylabel('logfoldchanges')

ax.axhline(0, color='gray', linestyle='--', lw=0.8)

for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis='x', length=0)

plt.tight_layout()
plt.savefig('IL1B_S100A8_CD16LST1_DEGs.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

### Functional enrichment analysis

In [ ]:
# GO (for Fig. 2G)
subcluster_configs = [
    ('DEGs_Mono_CD14_IL1B_vs_Other.csv', 'IL1B_Mono_GO_UP.xlsx'),
    ('DEGs_Mono_CD14_S100A8_vs_Other.csv', 'S100A8_Mono_GO_UP.xlsx'),
    ('DEGs_Mono_CD16_LST1_vs_Other.csv', 'CD16_Mono_GO_UP.xlsx')
]

for deg_file, out_file in subcluster_configs:
    # Input DEG file
    deg_df = pd.read_csv(deg_file)
    
    # filter for upregulated DEGs
    deg_up = deg_df[
        (deg_df['logfoldchanges'] > 0.5) & 
        (deg_df['pvals_adj'] < 0.05)
    ]['names'].tolist()

    # Run GO
    go_up = gp.enrichr(
        gene_list=deg_up,
        gene_sets=['GO_Biological_Process_2023'],
        organism='Human',
        outdir=None
    )
    
    # output GO results as dataframe
    go_up_df = go_up.results

    # Filter for significant terms
    go_up_df = go_up_df[go_up_df['Adjusted P-value'] < 0.05]

    # sort genes in 'Genes' column by LFC
    gene_rank_dict = dict(zip(deg_df['names'], deg_df['logfoldchanges']))

    def sort_genes_by_lfc(gene_string, rank_dict):
        if pd.isna(gene_string): return gene_string
        genes = gene_string.split(';')
        genes_sorted = sorted(genes, key=lambda x: rank_dict.get(x, 0), reverse=True)
        return ';'.join(genes_sorted)

    go_up_df['Genes'] = go_up_df['Genes'].apply(lambda x: sort_genes_by_lfc(x, gene_rank_dict))

    # Save
    go_up_df.to_excel(out_file, index=False)

In [ ]:
# KEGG (for Fig. S2E)
subcluster_configs = [
    ('DEGs_Mono_CD14_IL1B_vs_Other.csv', 'IL1B_Mono_KEGG_UP.xlsx'),
    ('DEGs_Mono_CD14_S100A8_vs_Other.csv', 'S100A8_Mono_KEGG_UP.xlsx'),
    ('DEGs_Mono_CD16_LST1_vs_Other.csv', 'CD16_Mono_KEGG_UP.xlsx')
]

for deg_file, out_file in subcluster_configs:
    # Input DEG file
    deg_df = pd.read_csv(deg_file)
    
    # filter for upregulated DEGs
    deg_up = deg_df[
        (deg_df['logfoldchanges'] > 0.5) & 
        (deg_df['pvals_adj'] < 0.05)
    ]['names'].tolist()

    # Run kegg
    kegg_up = gp.enrichr(
        gene_list=deg_up,
        gene_sets=['KEGG_2021_Human'],
        organism='Human',
        outdir=None
    )
    
    # output kegg results as dataframe
    kegg_up_df = kegg_up.results

    # Filter for significant terms
    kegg_up_df = kegg_up_df[kegg_up_df['Adjusted P-value'] < 0.05]

    # sort genes in 'Genes' column by LFC
    gene_rank_dict = dict(zip(deg_df['names'], deg_df['logfoldchanges']))

    def sort_genes_by_lfc(gene_string, rank_dict):
        if pd.isna(gene_string): return gene_string
        genes = gene_string.split(';')
        genes_sorted = sorted(genes, key=lambda x: rank_dict.get(x, 0), reverse=True)
        return ';'.join(genes_sorted)

    kegg_up_df['Genes'] = kegg_up_df['Genes'].apply(lambda x: sort_genes_by_lfc(x, gene_rank_dict))

    # Save
    kegg_up_df.to_excel(out_file, index=False)

### Functional enrichment visualization

#### GO
---
Fig. 2G

In [ ]:
# Barplot for selected GO terms of IL1B monocytes
import matplotlib.colors as mcolors

il1b_mono_go_df = pd.read_excel('IL1B_Mono_GO_UP.xlsx')

# Define selected GO terms
selected_terms = [
    'Positive Regulation Of Cytokine Production (GO:0001819)',
    'Positive Regulation Of Acute Inflammatory Response (GO:0002675)',
    'Chemokine-Mediated Signaling Pathway (GO:0070098)',
    'Cellular Response To Interleukin-1 (GO:0071347)'
]

# Filter for selected terms
il1b_barplot_df = il1b_mono_go_df[il1b_mono_go_df['Term'].isin(selected_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

il1b_barplot_df['Overlap_decimal'] = il1b_barplot_df['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
il1b_barplot_df = il1b_barplot_df.sort_values('Overlap_decimal', ascending=True)

# Define fixed color
colors = '#447296'

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=il1b_barplot_df['Term'],
    width=il1b_barplot_df['Overlap_decimal'],
    color=colors
)

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('il1b_go_up.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Barplot for selected GO terms of S100A8 monocytes
s100a8_mono_go_df = pd.read_excel('S100A8_Mono_GO_UP.xlsx')

# Define selected GO terms
selected_terms = [
    'Monocyte Activation (GO:0042117)',
    'Macrophage Activation (GO:0042116)',
    'Cell-Matrix Adhesion (GO:0007160)',
    'Extracellular Matrix Organization (GO:0030198)'
]

# Filter for selected terms
s100a8_barplot_df = s100a8_mono_go_df[s100a8_mono_go_df['Term'].isin(selected_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

s100a8_barplot_df['Overlap_decimal'] = s100a8_barplot_df['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
s100a8_barplot_df = s100a8_barplot_df.sort_values('Overlap_decimal', ascending=True)

# Define fixed color
colors = '#DB9B6D'

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=s100a8_barplot_df['Term'],
    width=s100a8_barplot_df['Overlap_decimal'],
    color=colors
)

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('s100a8_go_up.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Barplot for selected GO terms of CD16 monocytes
cd16_mono_go_df = pd.read_excel('CD16_Mono_GO_UP.xlsx')

# Define selected GO terms
selected_terms = [
    'Intrinsic Apoptotic Signaling Pathway (GO:0097193)',
    'Fc-gamma Receptor Signaling Pathway (GO:0038094)',
    'Regulation Of Granulocyte Differentiation (GO:0030852)',
    'Toll-Like Receptor 3 Signaling Pathway (GO:0034138)'
]

# Filter for selected terms
cd16_barplot_df = cd16_mono_go_df[cd16_mono_go_df['Term'].isin(selected_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

cd16_barplot_df['Overlap_decimal'] = cd16_barplot_df['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
cd16_barplot_df = cd16_barplot_df.sort_values('Overlap_decimal', ascending=True)

# Define fixed color
colors = '#4F3D80'

# plot
fig, ax = plt.subplots(figsize=(12,4))
bars = ax.barh(
    y=cd16_barplot_df['Term'],
    width=cd16_barplot_df['Overlap_decimal'],
    color=colors
)

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('cd16_go_up.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

#### KEGG
---
Fig. S2E

In [ ]:
# Barplot for selected KEGG terms of IL1B monocytes
il1b_mono_kegg_df = pd.read_excel('IL1B_Mono_KEGG_UP.xlsx')

# Define selected kegg terms
selected_terms = [
    'TNF signaling pathway',
    'NF-kappa B signaling pathway',
    'C-type lectin receptor signaling pathway',
    'IL-17 signaling pathway',
    'MAPK signaling pathway'
]

# Filter for selected terms
il1b_barplot_df = il1b_mono_kegg_df[il1b_mono_kegg_df['Term'].isin(selected_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

il1b_barplot_df['Overlap_decimal'] = il1b_barplot_df['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
il1b_barplot_df = il1b_barplot_df.sort_values('Overlap_decimal', ascending=True)

# Define fixed color
colors = '#447296'

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=il1b_barplot_df['Term'],
    width=il1b_barplot_df['Overlap_decimal'],
    color=colors
)

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('il1b_kegg_up.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Barplot for selected kegg terms of S100A8 monocytes
s100a8_mono_kegg_df = pd.read_excel('S100A8_Mono_KEGG_UP.xlsx')

# Define selected kegg terms
selected_terms = [
    'Hematopoietic cell lineage',
    'Complement and coagulation cascades',
    'ECM-receptor interaction'
]

# Filter for selected terms
s100a8_barplot_df = s100a8_mono_kegg_df[s100a8_mono_kegg_df['Term'].isin(selected_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

s100a8_barplot_df['Overlap_decimal'] = s100a8_barplot_df['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
s100a8_barplot_df = s100a8_barplot_df.sort_values('Overlap_decimal', ascending=True)

# Define fixed color
colors = '#DB9B6D'

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=s100a8_barplot_df['Term'],
    width=s100a8_barplot_df['Overlap_decimal'],
    color=colors
)

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('s100a8_kegg_up.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Barplot for selected kegg terms of CD16 monocytes
cd16_mono_kegg_df = pd.read_excel('CD16_Mono_KEGG_UP.xlsx')

# Define selected kegg terms
selected_terms = [
    'Apoptosis',
    'Antigen processing and presentation',
    'NOD-like receptor signaling pathway'
]

# Filter for selected terms
cd16_barplot_df = cd16_mono_kegg_df[cd16_mono_kegg_df['Term'].isin(selected_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

cd16_barplot_df['Overlap_decimal'] = cd16_barplot_df['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
cd16_barplot_df = cd16_barplot_df.sort_values('Overlap_decimal', ascending=True)

# Define fixed color
colors = '#4F3D80'

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=cd16_barplot_df['Term'],
    width=cd16_barplot_df['Overlap_decimal'],
    color=colors
)

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('cd16_kegg_up.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

## COVID19-specific DEGs

In [ ]:
# Subset adata_m to only COVID19 groups
covid_groups = ['COVID19_pro_Severe', 'COVID19_pro_Moderate', 'COVID19_con_Severe', 'COVID19_con_Moderate']
adata_covid = adata_m[adata_m.obs['group'].isin(covid_groups)].copy()

# Drop unwanted subclusters
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']
adata_covid = adata_covid[~adata_covid.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

### COVID-19 DEGs
---
Table S5d

In [ ]:
# Create a temporary object for DEG calculation
adata_deg = sc.AnnData(
    X = adata_covid.raw.X.copy(),
    obs = adata_covid.obs.copy(),
    var = adata_covid.raw.var.copy()
)

sc.pp.normalize_total(adata_deg, target_sum=1e4)
sc.pp.log1p(adata_deg)

# DEGs: pro vs con
sc.tl.rank_genes_groups(
    adata_deg,
    groupby='stage',
    reference = 'COVID19_con',
    groups = ['COVID19_pro'],
    method='wilcoxon',
    use_raw=False
)

# Extract, filter and save
m_pro_deg_df = sc.get.rank_genes_groups_df(adata_deg, group='COVID19_pro')
m_pro_deg_df = m_pro_deg_df[m_pro_deg_df['pvals_adj'] < 0.05]
m_pro_deg_df.to_csv('DEG_COVID19_pro_vs_con.csv', index=False)

# DEGs: Severe vs Moderate
adata_deg = adata_deg[adata_deg.obs['group'].isin(['COVID19_pro_Severe', 'COVID19_pro_Moderate'])]
sc.tl.rank_genes_groups(
    adata_deg,
    groupby='group',
    reference = 'COVID19_pro_Moderate',
    groups = ['COVID19_pro_Severe'],
    method='wilcoxon',
    use_raw=False
)
s_pro_severe_deg_df = sc.get.rank_genes_groups_df(adata_deg, group='COVID19_pro_Severe')
s_pro_severe_deg_df = s_pro_severe_deg_df[s_pro_severe_deg_df['pvals_adj'] < 0.05]
s_pro_severe_deg_df.to_csv('DEG_COVID19_Severe_vs_Moderate.csv', index=False)

### COVID-19 DEG volcano plot
---
Fig. 4D

In [ ]:
# bidirectional DEG plot 
# Input files: DEG_COVID19_pro_vs_con.csv and DEG_COVID19_Severe_vs_Moderate.csv

from adjustText import adjust_text

# read data
df1 = pd.read_csv('DEG_COVID19_pro_vs_con.csv').assign(group='pro_vs_con')
df2 = pd.read_csv('DEG_COVID19_Severe_vs_Moderate.csv').assign(group='S_vs_M')
df = pd.concat([df1, df2], ignore_index=True)

# filter for pvals_adj < 0.05
df = df[df['pvals_adj'] < 0.05].copy()
# Filter for logfoldchanges magnitude > 0.5 (UP > 0.5, DOWN < -0.5)
df = df[np.abs(df['logfoldchanges']) > 0.5].copy()

# label
label = {
    'pro_vs_con': [
        # up
        'IFI27', 'IL1R2', 'IL6', 'ISG15', 'IFITM3', 'RETN', 'S100A12', 'IFITM1',
        # down
        'IL1B', 'TNF', 'HLA-DRA', 'HLA-DPB1', 'HLA-DPA1', 'CSF1R'],
    'S_vs_M': [
        # UP
        'IL1R2', 'IL1A', 'CCL20', 'CCL4', 'CXCL3', 'IL6', 'IL1B', 'TNF', 'S100A12', 
        # DOWN
        'HLA-DQB2', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DRB5', 'CD74']
}

# point size
df['point_size'] = np.log1p(np.abs(df['scores']))

# assign labels
df['label'] = df.apply(
    lambda r: r['names'] if r['names'] in label[r['group']] else None,
    axis=1
)

# group order
group_order = ['pro_vs_con', 'S_vs_M']
x_map = {g: i for i, g in enumerate(group_order)}
df['x'] = df['group'].map(x_map)

# grey background boxes
shades = (
    df.groupby('group')['logfoldchanges']
    .agg(['min', 'max'])
    .reset_index()
)
shades['x'] = shades['group'].map(x_map)

# colors
# Red for Up, Blue for Down ONLY if labeled, otherwise darker gray
df['color'] = np.where(
    df['label'].notna(),
    np.where(df['logfoldchanges'] > 0, 'firebrick', 'steelblue'),
    "#b3b1b1ff"
)

# jitter
rng=np.random.default_rng(seed=1)
df['x_jitter'] = df['x'] + rng.uniform(-0.3, 0.3, size=len(df))

# Sort dataframe so labeled (colored) points are plotted on top
df['is_labeled'] = df['label'].notna()
df = df.sort_values('is_labeled', ascending=True)

# plot
fig, ax = plt.subplots(figsize=(8,12))

# plot grey backgrounds based on data range
for _, r in shades.iterrows():
    # Upregulated region background
    if r['max'] > 0.5:
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, 0.5), 
                width=0.7,
                height=r['max'] - 0.5 + 0.5,
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )
    # Downregulated region background
    if r['min'] < -0.5:
        y_start = r['min'] - 0.5
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, y_start),
                width=0.7,
                height=-0.5 - y_start, # Height from bottom to threshold -0.5
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )

# scatter points
ax.scatter(
    df['x_jitter'],
    df['logfoldchanges'],
    s=df['point_size'] * 50,
    c=df['color'],
    alpha=0.8,
    zorder=1
)
# gene labels
texts=[]
for _, r in df.dropna(subset=['label']).iterrows():
    texts.append(
        ax.text(
            r['x_jitter'],
            r['logfoldchanges'],
            r['label'],
            fontsize=8,
            color='black'
        )
    )
adjust_text(
    texts,
    ax=ax,
    only_move={'points':'y', 'texts':'y'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8),
    lim=200
)

# axes
ax.set_xlim(-0.6, len(group_order) - 0.4)

# Set dynamic symmetric y limits
y_abs_max = max(abs(df['logfoldchanges'].min()), abs(df['logfoldchanges'].max())) + 1
ax.set_ylim(-y_abs_max, y_abs_max)

# Add group labels on x axis
ax.set_xticks(list(x_map.values()))
ax.set_xticklabels(list(x_map.keys()))
ax.set_xlabel('')
ax.set_ylabel('logfoldchanges')

# Add 0 line
ax.axhline(0, color='gray', linestyle='--', lw=0.8)

for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis='x', length=0)

plt.tight_layout()
plt.savefig('pro_vs_con_vs_S_vs_M_DEGs.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

### Enrichment (pro vs con)
---
Fig. S4B

In [ ]:
covid19_disease_deg = pd.read_csv('DEG_COVID19_pro_vs_con.csv')

In [ ]:
import gseapy as gp
import pandas as pd

# Precompute gene rank dictionary
gene_rank_dict = dict(zip(covid19_disease_deg['names'], covid19_disease_deg['logfoldchanges']))

def sort_genes_by_lfc(gene_string):
    if pd.isna(gene_string):
        return gene_string
    genes = gene_string.split(';')
    genes_sorted = sorted(genes, key=lambda x: gene_rank_dict.get(x, 0), reverse=True)
    return ';'.join(genes_sorted)

# Define the two directions: (label, filter_condition, output_filename)
directions = [
    ('up', (covid19_disease_deg['logfoldchanges'] > 0.5), 'covid19_disease_up_go.xlsx'),
    ('down', (covid19_disease_deg['logfoldchanges'] < -0.5), 'covid19_disease_down_go.xlsx')
]

for label, lfc_condition, outfile in directions:
    # Filter for significant DEGs in this direction
    sig_genes = covid19_disease_deg[
        (covid19_disease_deg['pvals_adj'] < 0.05) & lfc_condition
    ]['names'].tolist()
    
    # Run GO enrichment
    go_result = gp.enrichr(
        gene_list=sig_genes,
        gene_sets=['GO_Biological_Process_2023'],
        organism='Human',
        outdir=None
    )
    go_df = go_result.results
    
    # Keep only significant terms (Adjusted p-value < 0.05)
    go_df = go_df[go_df['Adjusted P-value'] < 0.05]
    
    # Sort genes within each term by logFC
    go_df['Genes'] = go_df['Genes'].apply(sort_genes_by_lfc)
    
    # Save to Excel
    go_df.to_excel(outfile, index=False)

In [ ]:
# disease up barplot (Fig. S4B)
covid19_disease_go_up = pd.read_excel('covid19_disease_up_go.xlsx')

key_terms = [
    'Defense Response To Virus (GO:0051607)',
    'Regulation Of Tumor Necrosis Factor Production (GO:0032680)',
    'Response To Interferon-Alpha (GO:0035455)',
    'Neutrophil Chemotaxis (GO:0030593)',
    'Inflammatory Response (GO:0006954)',
    'Response To Cytokine (GO:0034097)',
    'Negative Regulation Of Viral Genome Replication (GO:0045071)'
]

# Filter for selected terms
covid19_disease_go_up = covid19_disease_go_up[covid19_disease_go_up['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

covid19_disease_go_up['Overlap_decimal'] = covid19_disease_go_up['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
covid19_disease_go_up = covid19_disease_go_up.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
covid19_disease_go_up['neg_log_padj'] = -np.log10(covid19_disease_go_up['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('OrRd')
norm = mcolors.Normalize(vmin=covid19_disease_go_up['neg_log_padj'].min(), vmax=covid19_disease_go_up['neg_log_padj'].max())
colors = cmap(norm(covid19_disease_go_up['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=covid19_disease_go_up['Term'],
    width=covid19_disease_go_up['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('covid19_disease_go_up.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# disease down barplot (Fig. S4B)
covid19_disease_go_down = pd.read_excel('covid19_disease_down_go.xlsx')

key_terms = [
    'MHC Class II Protein Complex Assembly (GO:0002399)',
    'Peptide Antigen Assembly With MHC Class II Protein Complex (GO:0002503)',
    'Immunoglobulin Mediated Immune Response (GO:0016064)',
    'Antigen Processing And Presentation Of Exogenous Peptide Antigen Via MHC Class II (GO:0019886)',
    'Antigen Processing And Presentation Of Exogenous Peptide Antigen (GO:0002478)'
]

# Filter for selected terms
covid19_disease_go_down = covid19_disease_go_down[covid19_disease_go_down['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

covid19_disease_go_down['Overlap_decimal'] = covid19_disease_go_down['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
covid19_disease_go_down = covid19_disease_go_down.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
covid19_disease_go_down['neg_log_padj'] = -np.log10(covid19_disease_go_down['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('Blues')
norm = mcolors.Normalize(vmin=covid19_disease_go_down['neg_log_padj'].min(), vmax=covid19_disease_go_down['neg_log_padj'].max())
colors = cmap(norm(covid19_disease_go_down['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=covid19_disease_go_down['Term'],
    width=covid19_disease_go_down['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('covid19_disease_go_down.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

### Enrichment (severity)
---
Fig. S4C

In [ ]:
covid19_severity_deg = pd.read_csv('DEG_COVID19_Severe_vs_Moderate.csv')

In [ ]:
import gseapy as gp
import pandas as pd

# Precompute gene rank dictionary
gene_rank_dict = dict(zip(covid19_severity_deg['names'], covid19_severity_deg['logfoldchanges']))

def sort_genes_by_lfc(gene_string):
    if pd.isna(gene_string):
        return gene_string
    genes = gene_string.split(';')
    genes_sorted = sorted(genes, key=lambda x: gene_rank_dict.get(x, 0), reverse=True)
    return ';'.join(genes_sorted)

# Define the two directions: (label, filter_condition, output_filename)
directions = [
    ('up', (covid19_severity_deg['logfoldchanges'] > 0.5), 'covid19_severity_up_go.xlsx'),
    ('down', (covid19_severity_deg['logfoldchanges'] < -0.5), 'covid19_severity_down_go.xlsx')
]

for label, lfc_condition, outfile in directions:
    # Filter for significant DEGs in this direction
    sig_genes = covid19_severity_deg[
        (covid19_severity_deg['pvals_adj'] < 0.05) & lfc_condition
    ]['names'].tolist()
    
    # Run GO enrichment
    go_result = gp.enrichr(
        gene_list=sig_genes,
        gene_sets=['GO_Biological_Process_2023'],
        organism='Human',
        outdir=None
    )
    go_df = go_result.results
    
    # Keep only significant terms (Adjusted p-value < 0.05)
    go_df = go_df[go_df['Adjusted P-value'] < 0.05]
    
    # Sort genes within each term by logFC
    go_df['Genes'] = go_df['Genes'].apply(sort_genes_by_lfc)
    
    # Save to Excel
    go_df.to_excel(outfile, index=False)

In [ ]:
# barplot for severity up (Fig. S4C)
from matplotlib import colors as mcolors

# severity up
covid19_severity_go_up = pd.read_excel('covid19_severity_up_go.xlsx')

key_terms = [
    'Regulation Of Inflammatory Response (GO:0050727)',
    'Cytokine-Mediated Signaling Pathway (GO:0019221)',
    'Positive Regulation Of Acute Inflammatory Response (GO:0002675)',
    'Granulocyte Chemotaxis (GO:0071621)',
    'Neutrophil Chemotaxis (GO:0030593)',
    'Regulation Of B Cell Activation (GO:0050864)',
    'Regulation Of Interleukin-6 Production (GO:0032675)'
]

# Filter for selected terms
covid19_severity_go_up = covid19_severity_go_up[covid19_severity_go_up['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

covid19_severity_go_up['Overlap_decimal'] = covid19_severity_go_up['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
covid19_severity_go_up = covid19_severity_go_up.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
covid19_severity_go_up['neg_log_padj'] = -np.log10(covid19_severity_go_up['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('OrRd')
norm = mcolors.Normalize(vmin=covid19_severity_go_up['neg_log_padj'].min(), vmax=covid19_severity_go_up['neg_log_padj'].max())
colors = cmap(norm(covid19_severity_go_up['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=covid19_severity_go_up['Term'],
    width=covid19_severity_go_up['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('covid19_severity_go_up.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# barplot for severity down (Fig. S4C)
covid19_severity_go_down = pd.read_excel('covid19_severity_down_go.xlsx')

key_terms = [
    'MHC Class II Protein Complex Assembly (GO:0002399)',
    'Antigen Processing And Presentation Of Peptide Antigen Via MHC Class II (GO:0002495)',
    'Positive Regulation Of Lymphocyte Activation (GO:0051251)',
    'Positive Regulation Of Leukocyte Cell-Cell Adhesion (GO:1903039)',
    'Positive Regulation Of T Cell Activation (GO:0050870)',
    'Positive Regulation Of I-kappaB kinase/NF-kappaB Signaling (GO:0043123)'
]

# Filter for selected terms
covid19_severity_go_down = covid19_severity_go_down[covid19_severity_go_down['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

covid19_severity_go_down['Overlap_decimal'] = covid19_severity_go_down['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
covid19_severity_go_down = covid19_severity_go_down.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
covid19_severity_go_down['neg_log_padj'] = -np.log10(covid19_severity_go_down['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('Blues')
norm = mcolors.Normalize(vmin=covid19_severity_go_down['neg_log_padj'].min(), vmax=covid19_severity_go_down['neg_log_padj'].max())
colors = cmap(norm(covid19_severity_go_down['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=covid19_severity_go_down['Term'],
    width=covid19_severity_go_down['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('covid19_severity_go_down.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

## SLE-specific DEGs

In [83]:
# subset adata_m to exclude specific subclusters that are sample-biased
clusters_to_exclude = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']
adata_m_sle = adata_m[~adata_m.obs['myeloid_anno'].isin(clusters_to_exclude)].copy()

# Further filter to only include desired groups (for cross-disease and cross-SLE-groups analyses)
groups_to_retain = ['CAR-T_CRS_pro_Severe', 'COVID19_pro_Severe', 'SLE_Severe', 'SLE_Moderate']
adata_m_sle = adata_m_sle[adata_m_sle.obs['group'].isin(groups_to_retain)].copy()

### SLE_vs_rest_disease and SLE_Severe_vs_SLE_Moderate
---
Fig. 5A, Fig S5A, Table S6b

In [ ]:
# SLE_vs_rest_disease
compare_groups = ['CAR-T_CRS_pro_Severe', 'COVID19_pro_Severe', 'SLE_Severe']

# Subset to the groups to compare
adata_sub = adata_m_sle[adata_m_sle.obs['group'].isin(compare_groups)].copy()

# Create comparison group
adata_sub.obs['comp_target'] = adata_sub.obs['group'].apply(lambda x: 'SLE_Severe' if x == 'SLE_Severe' else 'REST').astype('category')

# Run DE
sc.tl.rank_genes_groups(adata_sub, groupby='comp_target', groups=['SLE_Severe'], reference='REST', method='wilcoxon', use_raw=False)

# Get results, filter and save (Table S6b)
df_res = sc.get.rank_genes_groups_df(adata_sub, group='SLE_Severe')
df_res = df_res[df_res['pvals_adj'] < 0.05]
df_res.to_csv('DEGs_SLE_Severe_vs_REST.csv', index=False) # Table S6b

In [ ]:
# SLE_Severe_vs_SLE_Moderate
compare_groups = ['SLE_Severe', 'SLE_Moderate']

# Subset to the groups to compare
adata_sub = adata_m_sle[adata_m_sle.obs['group'].isin(compare_groups)].copy()

# Create comparison group
adata_sub.obs['comp_target'] = adata_sub.obs['group'].apply(lambda x: 'SLE_Severe' if x == 'SLE_Severe' else 'SLE_Moderate').astype('category')

# Run DE
sc.tl.rank_genes_groups(adata_sub, groupby='comp_target', groups=['SLE_Severe'], reference='SLE_Moderate', method='wilcoxon', use_raw=False)

# Get results, filter and save
df_res = sc.get.rank_genes_groups_df(adata_sub, group='SLE_Severe')
df_res = df_res[df_res['pvals_adj'] < 0.05]
df_res.to_csv('DEGs_SLE_Severe_vs_SLE_Moderate.csv', index=False) # Table S6b

In [ ]:
# volcano DEG plot (Fig. S5A)
# Input file: DEGs_SLE_Severe_vs_REST.csv and DEGs_SLE_Severe_vs_SLE_Moderate.csv

from adjustText import adjust_text

# Read data
df1 = pd.read_csv('DEGs_SLE_Severe_vs_REST.csv').assign(group='SLE_vs_REST')
df2 = pd.read_csv('DEGs_SLE_Severe_vs_SLE_Moderate.csv').assign(group='Severe_vs_Moderate')
df = pd.concat([df1, df2], ignore_index=True)

# filter for pvals_adj < 0.05
df = df[df['pvals_adj'] < 0.05].copy()
# Filter for logfoldchanges magnitude > 0.5 (UP > 0.5)
df = df[df['logfoldchanges'] > 0.5].copy()

# label
label = {
    'SLE_vs_REST': [
        # up
        'ISG15', 'IFI27', 'IFI6', 'IFITM3', 'HLA-DRA', 'CD74', 'FCER1G', 'S100A8', 'S100A9', 'NACA2', 'COX4I1',
    ],
    'Severe_vs_Moderate': [
        # UP
        'ISG15', 'IFI27', 'IFITM3', 'IFI6', 'S100A9', 'S100A8', 'CCL2', 'FCER1G', 'FCGR1A']
}

# point size
# Use 'scores' for point size
df['point_size'] = np.log1p(np.abs(df['scores']))

# assign labels
df['label'] = df.apply(
    lambda r: r['names'] if r['names'] in label[r['group']] else None,
    axis=1
)

# group order
group_order = ['SLE_vs_REST', 'Severe_vs_Moderate']
x_map = {g: i for i, g in enumerate(group_order)}
df['x'] = df['group'].map(x_map)

# grey background boxes
shades = (
    df.groupby('group')['logfoldchanges']
    .agg(['min', 'max'])
    .reset_index()
)
shades['x'] = shades['group'].map(x_map)

# colors
# Red for Up ONLY if labeled, otherwise darker gray
df['color'] = np.where(
    df['label'].notna(),
    'firebrick',
    "#b3b1b1ff"
)

# jitter
rng=np.random.default_rng(seed=1)
df['x_jitter'] = df['x'] + rng.uniform(-0.3, 0.3, size=len(df))

# sort dataframe so labeled (colored) points are plotted on top
df['is_labeled'] = df['label'].notna()
df = df.sort_values('is_labeled', ascending=True)

# plotting
fig, ax = plt.subplots(figsize=(8,12))

# plot grey backgrounds based on data range
for _, r in shades.iterrows():
    # Upregulated region background
    if r['max'] > 0.5:
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, 0.5), 
                width=0.7,
                height=r['max'] - 0.5 + 0.5,
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )

# scatter points
ax.scatter(
    df['x_jitter'],
    df['logfoldchanges'],
    s=df['point_size'] * 50,
    c=df['color'],
    alpha=0.8,
    zorder=1
)
# gene labels
texts=[]
for _, r in df.dropna(subset=['label']).iterrows():
    texts.append(
        ax.text(
            r['x_jitter'],
            r['logfoldchanges'],
            r['label'],
            fontsize=8,
            color='black'
        )
    )
adjust_text(
    texts,
    ax=ax,
    only_move={'points':'y', 'texts':'y'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8),
    lim=200
)

# axes
ax.set_xlim(-0.6, len(group_order) - 0.4)

# Set dynamic y limits for upregulated
y_max = df['logfoldchanges'].max() + 1
ax.set_ylim(-0.5, y_max)

# Add group labels on X axis
ax.set_xticks(list(x_map.values()))
ax.set_xticklabels(list(x_map.keys()))
ax.set_xlabel('')
ax.set_ylabel('logfoldchanges')

# Add 0 line
ax.axhline(0, color='gray', linestyle='--', lw=0.8)

for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis='x', length=0)

plt.tight_layout()
plt.savefig('SLE_DEGs.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

#### Enrichment and visualization
---
Fig. 5A

In [ ]:
import pandas as pd
import gseapy as gp

files = {
    'SLE_Severe_vs_REST': 'DEGs_SLE_Severe_vs_REST.csv',
    'SLE_Severe_vs_SLE_Moderate': 'DEGs_SLE_Severe_vs_SLE_Moderate.csv'
}
directions = {'UP': 0.5, 'DOWN': -0.5}

for comp_name, file_name in files.items():
    # Read the DEGs file
    df = pd.read_csv(file_name)
    gene_rank_dict = dict(zip(df['names'], df['logfoldchanges']))
    
    for direction, threshold in directions.items():
        # Filter gene list based on direction
        if direction == 'UP':
            sig_genes = df[df['logfoldchanges'] > threshold]['names'].tolist()
            is_reversed = True  # Sort so highest positive LFC is first
        else:
            sig_genes = df[df['logfoldchanges'] < threshold]['names'].tolist()
            is_reversed = False # Sort so lowest negative LFC is first
            
        if not sig_genes:
            print(f"No significant {direction} genes for {comp_name}.")
            continue
            
        # Run GO Enrichment
        go_res = gp.enrichr(
            gene_list=sig_genes,
            gene_sets=['GO_Biological_Process_2023'],
            organism='Human',
            outdir=None
        )
        
        go_df = go_res.results
        go_df = go_df[go_df['Adjusted P-value'] < 0.05].copy()
        
        if go_df.empty:
            print(f"No significant GO terms for {comp_name} ({direction}).")
            continue

        # Sort genes in 'Genes' column by logFC
        def sort_genes_by_lfc(gene_string):
            if pd.isna(gene_string): return gene_string
            genes = gene_string.split(';')
            genes_sorted = sorted(genes, key=lambda x: gene_rank_dict.get(x, 0), reverse=is_reversed)
            return ';'.join(genes_sorted)

        go_df['Genes'] = go_df['Genes'].apply(sort_genes_by_lfc)

        # Save to excel
        output_file = f'{comp_name}_{direction}.xlsx'
        go_df.to_excel(output_file, index=False)
        print(f"Saved {output_file}")

Saved SLE_Severe_vs_REST_UP.xlsx
Saved SLE_Severe_vs_REST_DOWN.xlsx
Saved SLE_Severe_vs_SLE_Moderate_UP.xlsx
Saved SLE_Severe_vs_SLE_Moderate_DOWN.xlsx


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Visualization: barplot of selected key terms

# SLE_vs_Rest up
sle_up_go = pd.read_excel('SLE_Severe_vs_REST_UP.xlsx')

key_terms = [
    'Antigen Processing And Presentation Of Exogenous Peptide Antigen (GO:0002478)',
    'Positive Regulation Of T Cell Activation (GO:0050870)',
    'Inflammatory Response (GO:0006954)',
    'T Cell Migration (GO:0072678)'
]

# filter for key terms
sle_up_barplot = sle_up_go[sle_up_go['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

sle_up_barplot['Overlap_decimal'] = sle_up_barplot['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
sle_up_barplot = sle_up_barplot.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
sle_up_barplot['neg_log_padj'] = -np.log10(sle_up_barplot['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('OrRd')
norm = mcolors.Normalize(vmin=sle_up_barplot['neg_log_padj'].min(), vmax=sle_up_barplot['neg_log_padj'].max())
colors = cmap(norm(sle_up_barplot['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=sle_up_barplot['Term'],
    width=sle_up_barplot['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('sle_up_go.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Visualization: barplot of selected key terms

# severity up
ssle_up_go = pd.read_excel('SLE_Severe_vs_SLE_Moderate_UP.xlsx')

key_terms = [
    'Response To Interferon-Beta (GO:0035456)',
    'Response To Interferon-Alpha (GO:0035455)',
    'Response To Type II Interferon (GO:0034341)',
    'Regulation Of Tumor Necrosis Factor Production (GO:0032680)',
    'Cellular Response To Type I Interferon (GO:0071357)'
]

# filter for key terms
ssle_up_barplot = ssle_up_go[ssle_up_go['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

ssle_up_barplot['Overlap_decimal'] = ssle_up_barplot['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
ssle_up_barplot = ssle_up_barplot.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
ssle_up_barplot['neg_log_padj'] = -np.log10(ssle_up_barplot['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('OrRd')
norm = mcolors.Normalize(vmin=ssle_up_barplot['neg_log_padj'].min(), vmax=ssle_up_barplot['neg_log_padj'].max())
colors = cmap(norm(ssle_up_barplot['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=ssle_up_barplot['Term'],
    width=ssle_up_barplot['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('ssle_up_go.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

### CD16 monocyte DEGs in SLE_Severe
---
Fig. 5G, Fig. S5G, Table S6b

In [84]:
# Calculate DEGs of Mono_CD16_LST1 against other monocytes in SLE_Severe
adata_sle_severe = adata_m_sle[adata_m_sle.obs['group'] == 'SLE_Severe'].copy()
sc.tl.rank_genes_groups(adata_sle_severe, groupby='myeloid_anno', method='wilcoxon', use_raw=False)
deg_results = sc.get.rank_genes_groups_df(adata_sle_severe, group='Mono_CD16_LST1')
deg_results.to_csv('SLE_Severe_Mono_CD16_LST1_vs_Others_DEGs.csv', index=False) # Table S6b

ranking genes
    finished (0:00:25)


In [ ]:
# volcano DEG plot (Fig. S5G)
# Input file: SLE_Severe_Mono_CD16_LST1_vs_Others_DEGs.csv

from adjustText import adjust_text

# read data
df = pd.read_csv('SLE_Severe_Mono_CD16_LST1_vs_Others_DEGs.csv')

# filter for pvals_adj < 0.05
df = df[df['pvals_adj'] < 0.05].copy()
# Filter for logfoldchanges magnitude > 0.5
df = df[df['logfoldchanges'].abs() > 0.5].copy()

# label
label = [
    'FCGR3A', 'LST1', 'FCER1G', 'IFITM2', 'IFITM3', 'ISG15', 'TNF', 'IFNAR1', 'IFNAR2', 'CSF1R'
]

# point size
df['point_size'] = np.log1p(np.abs(df['scores']))

# assign labels
df['label'] = df['names'].apply(lambda x: x if x in label else None)

# group order
df['group'] = 'Mono_CD16_LST1'
group_order = ['Mono_CD16_LST1']
x_map = {g: i for i, g in enumerate(group_order)}
df['x'] = df['group'].map(x_map)

# grey background
shades = (
    df.groupby('group')['logfoldchanges']
    .agg(['min', 'max'])
    .reset_index()
)
shades['x'] = shades['group'].map(x_map)

# colors
def get_color(row):
    if pd.notna(row['label']):
        if row['logfoldchanges'] > 0:
            return 'firebrick'
        else:
            return 'steelblue'
    return "#b3b1b1ff"

df['color'] = df.apply(get_color, axis=1)

# jitter
rng=np.random.default_rng(seed=1)
df['x_jitter'] = df['x'] + rng.uniform(-0.3, 0.3, size=len(df))

# Sort dataframe so labeled (colored) points are plotted on top
df['is_labeled'] = df['label'].notna()
df = df.sort_values('is_labeled', ascending=True)

# plot
fig, ax = plt.subplots(figsize=(6, 12))

# plot grey backgrounds based on data range
for _, r in shades.iterrows():
    # Upregulated region background
    if r['max'] > 0.5:
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, 0.5), 
                width=0.7,
                height=r['max'] - 0.5 + 0.5,
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )
    # Downregulated region background
    if r['min'] < -0.5:
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, r['min'] - 0.5), 
                width=0.7,
                height=abs(r['min'] + 0.5) + 0.5,
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )

# scatter points
ax.scatter(
    df['x_jitter'],
    df['logfoldchanges'],
    s=df['point_size'] * 50,
    c=df['color'],
    alpha=0.8,
    zorder=1
)

# gene labels
texts=[]
for _, r in df.dropna(subset=['label']).iterrows():
    texts.append(
        ax.text(
            r['x_jitter'],
            r['logfoldchanges'],
            r['label'],
            fontsize=8,
            color='black'
        )
    )
adjust_text(
    texts,
    ax=ax,
    only_move={'points':'y', 'texts':'y'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8),
    lim=200
)

# axes
ax.set_xlim(-0.6, len(group_order) - 0.4)

# Set dynamic y limits
y_max = df['logfoldchanges'].max() + 1
y_min = df['logfoldchanges'].min() - 1
ax.set_ylim(min(-0.5, y_min), max(0.5, y_max))

# Add group labels on x axis
ax.set_xticks(list(x_map.values()))
ax.set_xticklabels(list(x_map.keys()))
ax.set_xlabel('')
ax.set_ylabel('logfoldchanges')

# Add 0 line
ax.axhline(0, color='gray', linestyle='--', lw=0.8)

for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis='x', length=0)

plt.tight_layout()
plt.savefig('SLE_Severe_Mono_CD16_LST1_vs_Others_DEGs.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [86]:
# GO enrichment of CD16 Mono DEGs

# Read DEG file
deg_df = pd.read_csv('SLE_Severe_Mono_CD16_LST1_vs_Others_DEGs.csv')

# Filter DEGs by logfoldchange
deg_df_filtered = deg_df[deg_df['logfoldchanges'] > 0.5]

# Get list of upregulated genes
upregulated_genes = deg_df_filtered['names'].tolist()

# Perform GO enrichment analysis using gseapy
enr = gp.enrichr(
    gene_list=upregulated_genes,
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir=None,
    cutoff=0.05
)
# Convert results to DataFrame
enrichment_results = enr.results 
# Save results to CSV
enrichment_results.to_csv('SLE_Severe_CD16_vs_Others_DEGs_GO_enrichment.csv', index=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Visualization: barplot of selected key terms (Fig. 5G)

# CD16 GO terms
cd16_up_go = pd.read_csv('SLE_Severe_CD16_vs_Others_DEGs_GO_enrichment.csv')

key_terms = [
    'tumor necrosis factor-mediated signaling pathway (GO:0033209)',
    'Fc-gamma receptor signaling pathway (GO:0038094)',
    'Fc receptor mediated stimulatory signaling pathway (GO:0002431)',
    'type I interferon signaling pathway (GO:0060337)',
    'antigen processing and presentation of exogenous peptide antigen via MHC class II (GO:0019886)'
]

# filter for key terms
cd16_up_barplot = cd16_up_go[cd16_up_go['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

cd16_up_barplot['Overlap_decimal'] = cd16_up_barplot['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
cd16_up_barplot = cd16_up_barplot.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
cd16_up_barplot['neg_log_padj'] = -np.log10(cd16_up_barplot['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('OrRd')
norm = mcolors.Normalize(vmin=cd16_up_barplot['neg_log_padj'].min(), vmax=cd16_up_barplot['neg_log_padj'].max())
colors = cmap(norm(cd16_up_barplot['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=cd16_up_barplot['Term'],
    width=cd16_up_barplot['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('cd16_up_go.pdf', dpi=300, bbox_inches='tight')
plt.show()

# 8. Monocyte functional module scoring

Module scoring, scaling, statistical comparisons, and figure/table exports are downstream analyses of the processed myeloid object.

## Cross disease
---
Table S5a, Fig. 4A

In [97]:
# Only retain the three concrete monocyte subclusters
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']

adata_mo = adata_m[~adata_m.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

# groups to keep (Active diseases and HC)
groups_to_keep = ['CAR-T_CRS_pro', 'COVID19_pro', 'SLE', 'HC']
adata_mo = adata_mo[adata_mo.obs['stage'].isin(groups_to_keep)].copy()

# Remove No_CRS in CAR-T_CRS_pro
adata_mo = adata_mo[~((adata_mo.obs['stage'] == 'CAR-T_CRS_pro') & (adata_mo.obs['severity'] == 'No_CRS'))].copy()

In [98]:
# Define and calculate functional module scores
gene_dict = {
    'Emergency_myelopoiesis': ['MPO', 'LTF', 'VCAN', 'RETN', 'LCN2', 'S100A12', 'MMP8', 'OLR1'],
    'Immunoparalysis_down': ['HLA-DRB1', 'HLA-DRA', 'CD74', 'CIITA'],
    'Immunoparalysis_up': ['SOCS3', 'IRAK3', 'TNFAIP3', 'DUSP1']
}

for module, genes in gene_dict.items():
    valid_genes = [gene for gene in genes if gene in adata_mo.raw.var_names]
    if len(valid_genes) > 0:
        sc.tl.score_genes(adata_mo, gene_list=valid_genes, score_name=module, use_raw=False)
    else:
        print(f"Warning: No valid genes found for module {module}")

# calculate the combined immunoparalysis score (upregulated minus downregulated)
if 'Immunoparalysis_up' in adata_mo.obs.columns and 'Immunoparalysis_down' in adata_mo.obs.columns:
    adata_mo.obs['Immunoparalysis'] = adata_mo.obs['Immunoparalysis_up'] - adata_mo.obs['Immunoparalysis_down']

    # Remove the sub-module definitions from the dictionary to avoid looping over them downstream
    del gene_dict['Immunoparalysis_up']
    del gene_dict['Immunoparalysis_down']
    gene_dict['Immunoparalysis'] = ['HLA-DRB1', 'HLA-DRA', 'CD74', 'CIITA', 'SOCS3', 'IRAK3', 'TNFAIP3', 'DUSP1']

computing score 'Emergency_myelopoiesis'
    finished (0:00:03)
computing score 'Immunoparalysis_down'
    finished (0:00:03)
computing score 'Immunoparalysis_up'
    finished (0:00:03)


In [99]:
# Scale the scores for better visualization
# Use Z-score transformation
for module in gene_dict.keys():
    if module in adata_mo.obs.columns:
        # Calculate mean and standard deviation
        mean_val = adata_mo.obs[module].mean()
        std_val = adata_mo.obs[module].std()
        
        # Apply Z-score transformation
        adata_mo.obs[f'{module}_scaled'] = (adata_mo.obs[module] - mean_val) / std_val

In [100]:
# Define group order for consistent plotting
group_order = ['CAR-T_CRS_pro', 'COVID19_pro', 'SLE', 'HC']
adata_mo.obs['stage'] = pd.Categorical(adata_mo.obs['stage'], categories=group_order, ordered=True)

In [101]:
# Output per-sample means to an xlsx file in long format (metadata for Table S5a and Fig. 4A)
# Columns = sample_id, stage, Emergency_myelopoiesis_scaled, Immunoparalysis_scaled

# Compute per-sample means
sample_means = (
    adata_mo.obs[['sample_id', 'stage', 'Emergency_myelopoiesis', 'Immunoparalysis']]
    .groupby(['sample_id', 'stage'], observed=True)[['Emergency_myelopoiesis', 'Immunoparalysis']]
    .mean()
    .reset_index()
)

# Keep only rows where at least one metric exists
sample_means = sample_means.dropna(subset=['Emergency_myelopoiesis', 'Immunoparalysis'], how='all')

# Save to Excel 
sample_means.to_excel("sample_means_emergency_immunoparalysis.xlsx", index=False)

## Within COVID-19 groups
---
Table S5b, Fig. 4B, Fig. 4C, Fig. S4A, Fig. 4E, Fig. S4D

In [102]:
# Remove Mono_CD14_IFI44 and Mono_CD14_CD16, because they are sample-biased
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']

adata_mo = adata_m[~adata_m.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

# groups to keep
groups_to_keep = ['COVID19_pro_Moderate', 'COVID19_pro_Severe', 'COVID19_con_Moderate', 'COVID19_con_Severe', 'HC_HC']
adata_mo = adata_mo[adata_mo.obs['group'].isin(groups_to_keep)].copy()

### Calculate monocyte functional scoring

In [103]:
# Define and calculate functional module scores

gene_dict = {
    'Emergency_myelopoiesis': ['MPO', 'LTF', 'VCAN', 'RETN', 'LCN2', 'S100A12', 'MMP8', 'OLR1'],
    'Type_I_IFN_response': ['ISG15', 'IFITM3', 'IFI44L', 'IFIT1', 'IFI6', 'MX1', 'SIGLEC1'],
    'Chemokine_response': ['CCR1', 'CCR2', 'CCR5', 'CXCR3', 'CXCR4', 'CX3CR1'],
    'Calprotectin': ['S100A8', 'S100A9'],
    'Glycolysis': ['HK2', 'PFKP', 'PKM', 'LDHA', 'SLC2A1', 'SLC2A3'],
    'Immunoparalysis_down': ['HLA-DRB1', 'HLA-DRA', 'CD74', 'CIITA'],
    'Immunoparalysis_up': ['SOCS3', 'IRAK3', 'TNFAIP3', 'DUSP1'],
    'Neutrophil_recruitment': ['CXCL1', 'CXCL2', 'CXCL3', 'CXCL5', 'CXCL6', 'CSF3', 'CSF2', 'S100A12'],
    'Antigen_presentation': ['HLA-DRA', 'HLA-DRB1', 'CD74', 'CIITA']
}

for module, genes in gene_dict.items():
    valid_genes = [gene for gene in genes if gene in adata_mo.raw.var_names]
    if len(valid_genes) > 0:
        sc.tl.score_genes(adata_mo, gene_list=valid_genes, score_name=module, use_raw=False)
    else:
        print(f"Warning: No valid genes found for module {module}")

# calculate the combined immunoparalysis score (upregulated minus downregulated)
if 'Immunoparalysis_up' in adata_mo.obs.columns and 'Immunoparalysis_down' in adata_mo.obs.columns:
    adata_mo.obs['Immunoparalysis'] = adata_mo.obs['Immunoparalysis_up'] - adata_mo.obs['Immunoparalysis_down']
        
    # Remove the sub-module definitions from the dictionary to avoid looping over them downstream
    del gene_dict['Immunoparalysis_up']
    del gene_dict['Immunoparalysis_down']
    gene_dict['Immunoparalysis'] = ['HLA-DRB1', 'HLA-DRA', 'CD74', 'CIITA', 'SOCS3', 'IRAK3', 'TNFAIP3', 'DUSP1']

computing score 'Emergency_myelopoiesis'
    finished (0:00:06)
computing score 'Type_I_IFN_response'
    finished (0:00:06)
computing score 'Chemokine_response'
    finished (0:00:06)
computing score 'Calprotectin'
    finished (0:00:06)
computing score 'Glycolysis'
    finished (0:00:06)
computing score 'Immunoparalysis_down'
    finished (0:00:06)
computing score 'Immunoparalysis_up'
    finished (0:00:06)
computing score 'Neutrophil_recruitment'
    finished (0:00:06)
computing score 'Antigen_presentation'
    finished (0:00:06)


In [104]:
# Scale the scores for better visualization
# Use Z-score transformation
for module in gene_dict.keys():
    if module in adata_mo.obs.columns:
        # Calculate mean and standard deviation
        mean_val = adata_mo.obs[module].mean()
        std_val = adata_mo.obs[module].std()
        
        # Apply Z-score transformation
        adata_mo.obs[f'{module}_scaled'] = (adata_mo.obs[module] - mean_val) / std_val

In [105]:
# Define group order for consistent plotting
group_order = ['COVID19_pro_Severe', 'COVID19_pro_Moderate', 'COVID19_con_Severe', 'COVID19_con_Moderate', 'HC_HC']
adata_mo.obs['group'] = pd.Categorical(adata_mo.obs['group'], categories=group_order, ordered=True)

In [106]:
# scale 'CS_score' by z score as well
score_col = 'CS_score'
mean_val = adata_mo.obs[score_col].mean()
std_val = adata_mo.obs[score_col].std()
adata_mo.obs[f'{score_col}_scaled'] = (adata_mo.obs[score_col] - mean_val) / std_val

### Ridge plots of monocyte functional scoring
---
Fig. 4C, Fig. S4A

In [ ]:
# Ridge plot for CS score across COVID19 groups, colored by mean group CS score (RdBu_r)

from scipy import stats
from matplotlib import cm, colors

score_col = 'CS_score_scaled'
df_list = []

for g in group_order:
    sub = adata_mo.obs[adata_mo.obs['group'] == g].copy()
    x = sub[score_col]

    median = np.median(x)
    mad = np.median(np.abs(x - median))
    if mad == 0:
        mad = 1e-9

    modified_z = 0.6745 * (x - median) / mad
    sub = sub[np.abs(modified_z) <= 3.5]
    df_list.append(sub)

df_filtered = pd.concat(df_list)

# Color palette based on mean CS score per group
group_means_cs = adata_mo.obs.groupby('group')[score_col].median()
norm = colors.Normalize(vmin=group_means_cs.min(), vmax=group_means_cs.max())
cmap = cm.get_cmap('RdBu_r')
palette_cs = {grp: cmap(norm(val)) for grp, val in group_means_cs.items()}

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})
g = sns.FacetGrid(
    df_filtered,
    row="group",
    hue="group",
    aspect=4,
    height=1,
    palette=palette_cs
)

g.map(
    sns.kdeplot,
    score_col,
    clip_on=False,
    fill=True,
    alpha=0.9,
    linewidth=1.5,
    bw_adjust=1.5,
    cut=0
)
g.map(plt.axhline, y=0, lw=1, clip_on=False, color='black')

g.figure.subplots_adjust(hspace=-0.5)
g.set_titles("")
g.set(yticks=[], ylabel="", xlabel=None)
g.despine(bottom=True, left=True)

plt.xlim(df_filtered[score_col].min(), df_filtered[score_col].max())
plt.savefig('RidgePlot_CS_score_scaled_COVID.pdf', bbox_inches='tight', dpi=300)
plt.show()
plt.close()

In [ ]:
# Ridge plots for functional modules across COVID19 groups
from matplotlib import cm, colors

for module_name in gene_dict.keys():
    score_col = f"{module_name}_scaled"
    if score_col not in adata_mo.obs.columns:
        print(f"Skipping {score_col}: column not found")
        continue

    print(f"Plotting and testing for module: {score_col}")

    df_list = []
    for g in group_order:
        sub = adata_mo.obs[adata_mo.obs['group'] == g].copy()
        x = sub[score_col]

        median = np.median(x)
        mad = np.median(np.abs(x - median))
        if mad == 0:
            mad = 1e-9

        modified_z = 0.6745 * (x - median) / mad
        sub = sub[np.abs(modified_z) <= 3.5]
        df_list.append(sub)

    df_filtered = pd.concat(df_list)

    group_means = adata_mo.obs.groupby('group')[score_col].mean()
    norm = colors.Normalize(vmin=group_means.min(), vmax=group_means.max())
    cmap = plt.cm.get_cmap('RdBu_r')
    palette = {g: cmap(norm(group_means[g])) for g in group_order}

    sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})
    g = sns.FacetGrid(
        df_filtered,
        row="group",
        hue="group",
        aspect=4,
        height=1,
        palette=palette
    )

    g.map(
        sns.kdeplot,
        score_col,
        clip_on=False,
        fill=True,
        alpha=0.9,
        linewidth=1.5,
        bw_adjust=1.5,
        cut=0
    )
    g.map(plt.axhline, y=0, lw=1, clip_on=False, color='black')

    g.figure.subplots_adjust(hspace=-0.5)
    g.set_titles("")
    g.set(yticks=[], ylabel="", xlabel=None)
    g.despine(bottom=True, left=True)

    plt.xlim(df_filtered[score_col].min(), df_filtered[score_col].max())
    plt.savefig(f'RidgePlot_{score_col}.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()

### Dataframe of monocyte functional module scoring
---
Table S5b

In [109]:

import re

# Add 'CS_score' to the list of modules to export
modules_to_export = list(gene_dict.keys()) + ['CS_score']

output_path = "COVID19_monocyte_sample_level_module_scores.xlsx"

def clean_excel_sheet_name(name):
    name = re.sub(r'[:\\/?*\[\]]', "_", name)
    return name[:31]

with pd.ExcelWriter(output_path) as writer:
    used_sheet_names = set()

    for module_name in modules_to_export:
        score_col = f"{module_name}_scaled"

        if score_col not in adata_mo.obs.columns:
            print(f"Skipping {score_col}: column not found")
            continue

        print(f"Exporting sample-level means for: {score_col}")

        sample_means = (
            adata_mo.obs[["sample_id", "group", score_col]]
            .dropna(subset=["sample_id", "group", score_col])
            .groupby(["sample_id", "group"], observed=True)[score_col]
            .mean()
            .reset_index()
            .rename(columns={score_col: "sample_level_mean_score"})
        )

        if "group_order" in globals():
            sample_means["group"] = pd.Categorical(
                sample_means["group"],
                categories=group_order,
                ordered=True
            )
            sample_means = sample_means.sort_values(["group", "sample_id"])

        # Generate valid and unique sheet name
        sheet_name = clean_excel_sheet_name(module_name)

        original_sheet_name = sheet_name
        i = 1
        while sheet_name in used_sheet_names:
            suffix = f"_{i}"
            sheet_name = original_sheet_name[:31 - len(suffix)] + suffix
            i += 1

        used_sheet_names.add(sheet_name)

        sample_means.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

print(f"Saved to: {output_path}")

Exporting sample-level means for: Emergency_myelopoiesis_scaled
Exporting sample-level means for: Type_I_IFN_response_scaled
Exporting sample-level means for: Chemokine_response_scaled
Exporting sample-level means for: Calprotectin_scaled
Exporting sample-level means for: Glycolysis_scaled
Exporting sample-level means for: Neutrophil_recruitment_scaled
Exporting sample-level means for: Antigen_presentation_scaled
Exporting sample-level means for: Immunoparalysis_scaled
Exporting sample-level means for: CS_score_scaled
Saved to: COVID19_monocyte_sample_level_module_scores.xlsx


### Dotplot of monocyte functional module genes
---
Fig. S4D

In [ ]:
# remove HC
adata_dp = adata_mo[adata_mo.obs['group'] != 'HC_HC'].copy()

# Generate separated dotplots for each module
dp = sc.pl.dotplot(
    adata_dp, 
    var_names = gene_dict,
    groupby = 'group',
    use_raw = False,
    cmap='OrRd',
    standard_scale='var',
    swap_axes=True,
    dot_min=0,
    figsize=(5,15),
    return_fig=True
)

dp.make_figure()

scatter_artist = dp.ax_dict['mainplot_ax'].collections[0]

original_sizes = scatter_artist.get_sizes()

min_size = 20
new_size = original_sizes.copy()
new_size[new_size > 0] += min_size

scatter_artist.set_sizes(new_size)

scatter_artist.set_linewidths(0)

dp.fig.savefig('COVID19_functional_module_genes.pdf', bbox_inches='tight', dpi=300)

plt.show()

### Correlation of immunoparalysis and CS score
---
Fig. 4E

In [ ]:
# Explore correlations between functional modules at the sample level

# Subset to COVID19_pro_Severe and COVID19_pro_Moderate
adata_covid = adata_dp[adata_dp.obs['group'].isin(['COVID19_pro_Severe', 'COVID19_pro_Moderate'])].copy()

# Extract sample-level module scores with outlier removal
sample_scores_dict = {}

for module in gene_dict.keys():
    score_col = f"{module}_scaled"
    if score_col in adata_covid.obs.columns:
        x = adata_covid.obs[score_col].dropna()
        median = np.median(x)
        mad = np.median(np.abs(x - median))
        if mad == 0:
            mad = 1e-9

        modified_z = 0.6745 * (x - median) / mad
        valid_indices = x.index[np.abs(modified_z) <= 3.5]

        sample_scores_dict[module] = adata_covid.obs.loc[valid_indices].groupby('sample_id')[score_col].mean()
# Convert to dataframe
df_sample_scores = pd.DataFrame(sample_scores_dict)

# Correlation matrix
corr_matrix = df_sample_scores.corr(method='pearson')

# Clustermap
g = sns.clustermap(
    corr_matrix,
    cmap='RdBu_r',
    annot=False,
    fmt=".2f",
    linewidths=0.5,
    vmin=-1, vmax=1,
    figsize=(8, 8)
)
g.fig.suptitle("Sample-Level Correlation of Functional Modules", y=1.05)
plt.savefig("Corr_All_Modules_Clustermap.pdf", bbox_inches='tight', dpi=300)
plt.show()
plt.close()

In [ ]:
# Correlation between Immunoparalysis and CS score (Fig. 4E)

# color palette
custom_palette = {
    'HC_HC': 'grey',
    'COVID19_con_Moderate': '#2ecc71',
    'COVID19_con_Severe': '#3498db',
    'COVID19_pro_Moderate': '#9b59b6',
    'COVID19_pro_Severe': '#e74c3c'
}

adata_covid = adata_mo[adata_mo.obs['group'].isin(['COVID19_pro_Severe', 'COVID19_pro_Moderate', 'COVID19_con_Severe', 'COVID19_con_Moderate', 'HC_HC'])].copy()

# Per sample mean of both scores
plot_df = adata_covid.obs[['group', 'sample_id', 'Immunoparalysis', 'CS_score']].copy()
plot_df = plot_df.groupby(['group', 'sample_id'], observed=True)[['Immunoparalysis', 'CS_score']].mean().reset_index()

# data cleaning
plot_df_clean = plot_df.dropna(subset=['Immunoparalysis', 'CS_score']).copy()

# Map scores to 0-1 range for better visualization
plot_df_clean['Immunoparalysis'] = (plot_df_clean['Immunoparalysis'] - plot_df_clean['Immunoparalysis'].min()) / (plot_df_clean['Immunoparalysis'].max() - plot_df_clean['Immunoparalysis'].min())
plot_df_clean['CS_score'] = (plot_df_clean['CS_score'] - plot_df_clean['CS_score'].min()) / (plot_df_clean['CS_score'].max() - plot_df_clean['CS_score'].min())

# calculate Centroids
centroids = (
    plot_df_clean
    .groupby('group', observed=True)[['Immunoparalysis', 'CS_score']]
    .mean()
    .reset_index()
)
centroid_map = centroids.set_index('group')

fig, ax = plt.subplots(figsize=(8, 7))

# plot the links (bottom layer)
# showing the general direction/trend of the group mean

for grp, sub in plot_df_clean.groupby('group', observed=True):
    if grp not in centroid_map.index: continue
    cx, cy = centroid_map.loc[grp, 'Immunoparalysis'], centroid_map.loc[grp, 'CS_score']
    color = custom_palette.get(grp, 'gray')
    
    for _, r in sub.iterrows():
        ax.plot(
            [r['Immunoparalysis'], cx],
            [r['CS_score'], cy],
            color=color,
            alpha=0.2,
            lw=2,
            zorder=1
        )

# plot the points (middle layer)
sns.scatterplot(
    data=plot_df_clean,
    x='Immunoparalysis',
    y='CS_score',
    hue='group',
    palette=custom_palette,
    alpha=0.7,
    edgecolor='white', # to distinguish overlapping points
    linewidth=0.5,
    s=100,
    ax=ax,
    legend=True,
    zorder=3
)

# plot centroids (top layer)
for i, row in centroids.iterrows():
    grp = row['group']
    color = custom_palette.get(grp, 'gray')
    cx, cy = row['Immunoparalysis'], row['CS_score']

    # White background circle
    ax.scatter(cx, cy, s=350, color='white', edgecolors='none', zorder=4, alpha=0.9)
    # Colored foreground circle
    ax.scatter(cx, cy, s=250, color=color, edgecolor='black', linewidth=1.5, zorder=5)

# Set axis labels and title
ax.set_xlabel('Immunoparalysis (scaled 0-1)', fontsize=13, labelpad=10)
ax.set_ylabel('CS Score (scaled 0-1)', fontsize=13, labelpad=10)

# Add axis origin lines
ax.axhline(0, color='black', lw=0.5, ls='--', alpha=0.2)
ax.axvline(0, color='black', lw=0.5, ls='--', alpha=0.2)

sns.despine(trim=True, offset=5)

plt.tight_layout()
plt.savefig('Immunoparalysis_vs_CS_score_centroids.pdf', dpi=300, bbox_inches='tight')
plt.show()

### Correlation of EM and immunoparalysis
---
Fig. 4B

In [114]:
# Subset to only COVID19_pro_severe
adata_pro_severe = adata_dp[adata_dp.obs['group'] == 'COVID19_pro_Severe'].copy()

In [ ]:
# Sample-level correlation between Emergency myelopoiesis and Immunoparalysis scores

score_cols = ['Emergency_myelopoiesis_scaled', 'Immunoparalysis_scaled']

sample_means = (
    adata_pro_severe.obs[['sample_id'] + score_cols]
    .dropna(subset=score_cols)
    .groupby('sample_id', observed=True)[score_cols]
    .mean()
    .reset_index()
)

if len(sample_means) > 2:
    r, p = stats.pearsonr(sample_means[score_cols[0]], sample_means[score_cols[1]])

    plt.figure(figsize=(5, 4))
    sns.regplot(
        data=sample_means,
        x=score_cols[1],
        y=score_cols[0],
        scatter_kws={'s': 50, 'alpha': 0.7},
        line_kws={'color': 'red'}
    )
    plt.title(f"Sample-level correlation\nPearson r = {r:.2f}, p = {p:.2e}")
    plt.xlabel('Immunoparalysis (Scaled)')
    plt.ylabel('Emergency Myelopoiesis (Scaled)')
    plt.tight_layout()
    plt.savefig("sample_level_Emergency_vs_Immunoparalysis.pdf", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Not enough samples for correlation analysis.")

## SLE IFN-I response score

### Calculate SLE IFN_I_response score
---
Table S6a, Fig. 5B, Fig. S5F

In [120]:
# Remove Mono_CD14_IFI44 and Mono_CD14_CD16, because they are sample-biased
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']

adata_sle = adata_m[~adata_m.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

# groups to keep
groups_to_keep = ['SLE_Severe', 'SLE_Moderate', 'HC_HC']
adata_sle = adata_sle[adata_sle.obs['group'].isin(groups_to_keep)].copy()

In [121]:
# Define and calculate functional module scores
gene_dict = {
    'Type_I_IFN_response': ['ISG15', 'IFITM3', 'IFI44L', 'IFIT1', 'IFI6', 'MX1', 'SIGLEC1']
}

for module, genes in gene_dict.items():
    valid_genes = [gene for gene in genes if gene in adata_sle.raw.var_names]
    if len(valid_genes) > 0:
        sc.tl.score_genes(adata_sle, gene_list=valid_genes, score_name=module, use_raw=False)
    else:
        print(f"Warning: No valid genes found for module {module}")

computing score 'Type_I_IFN_response'
    finished (0:00:01)


In [ ]:
# generate a dotplot for the Type I IFN response genes across subclusters in SLE_Severe (Fig. S5F)
sc.pl.dotplot(
    adata_sle[adata_sle.obs['group'] == 'SLE_Severe'],
    var_names=gene_dict['Type_I_IFN_response'],
    groupby='myeloid_anno',
    standard_scale='var',
    dot_min = 0,
    dot_max = 1,
    cmap='RdBu_r',
    save='SLE_Severe_Monocyte_IFN_I_Dotplot.pdf'
)

In [123]:
# Scale the scores for better visualization
# Use Z-score transformation
for module in gene_dict.keys():
    if module in adata_sle.obs.columns:
        # Calculate mean and standard deviation
        mean_val = adata_sle.obs[module].mean()
        std_val = adata_sle.obs[module].std()
        
        # Apply Z-score transformation
        adata_sle.obs[f'{module}_scaled'] = (adata_sle.obs[module] - mean_val) / std_val

print("Scaled scores (Z-score transformed) added to adata_sle.obs with suffix '_scaled'")

Scaled scores (Z-score transformed) added to adata_sle.obs with suffix '_scaled'


In [124]:
# Output xlsx with stacked sample-level IFN-I response values per group (For Table S6a and Fig. 5B)

sample_means_ifn = (
    adata_sle.obs[['sample_id', 'group', 'Type_I_IFN_response']]
    .dropna(subset=['sample_id', 'group', 'Type_I_IFN_response'])
    .groupby(['sample_id', 'group'], observed=True)['Type_I_IFN_response']
    .mean()
    .reset_index()
)

# define group order
groups = ['SLE_Severe', 'SLE_Moderate', 'HC_HC']

# Stack values in each column
stacked_cols = {
    g: sample_means_ifn.loc[sample_means_ifn['group'] == g, 'Type_I_IFN_response'].reset_index(drop=True)
    for g in groups
}

df_ifn_pairwise = pd.DataFrame(stacked_cols)

# Save
df_ifn_pairwise.to_excel('IFN_I_response_sample_means_by_group.xlsx', index=False)

### Differential IFN responsive modules between SLE and COVID19
---
Fig. S5C

In [ ]:
# Subset adata_m to SLE groups and COVID19_pro_Severe for direct comparison
adata_sub = adata_m[adata_m.obs['group'].isin(['SLE_Severe', 'SLE_Moderate', 'COVID19_pro_Severe'])].copy()

# Define genes of interest (IFN-I responsive genes in DEGs)
genes_of_interest = ['ISG15', 'IFITM3', 'IFI44L', 'IFI6', 'IFIT1', 'IFNAR2', 'IFNAR1', 'SIGLEC1', 'SIGLEC9', 'SIGLEC10', 'MX1', 'IRF1', 'IRF7']

# Plot gene expressions across groups
available_genes = [g for g in genes_of_interest if g in adata_sub.var_names]

missing_genes = sorted(set(genes_of_interest) - set(available_genes))
if missing_genes:
    print("Missing genes:", ", ".join(missing_genes))

# Extract expression and compute group-wise mean
expr_df = sc.get.obs_df(adata_sub, keys=available_genes, use_raw=False)
expr_df["group"] = adata_sub.obs["group"].values
mean_expr = expr_df.groupby("group")[available_genes].mean().T

# Ensure desired column order
col_order = [g for g in ['SLE_Severe', 'SLE_Moderate', 'COVID19_pro_Severe'] if g in mean_expr.columns]
mean_expr = mean_expr[col_order]

# Heatmap with row clustering + row-wise z-score
g = sns.clustermap(
    mean_expr,
    row_cluster=True,
    col_cluster=False,
    z_score=0,
    cmap="RdBu_r",
    center=0,
    figsize=(5, 7),
    linewidths=0.2,
    linecolor="white"
)

g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_ylabel("")
g.fig.suptitle("IFN-related gene expression: SLE_Severe vs COVID19_pro_Severe", y=1.02)

g.fig.savefig("SLE_vs_COVID19_IFN_genes_clustermap.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

### Correlation of IFN_I_response and CS score
---
Fig. 5C, Fig. S5C, Fig. S5D

In [126]:
# Subset to SLE, COVID19_pro_Severe, and HC for direct comparison
adata_sub = adata_m[adata_m.obs['group'].isin(['SLE_Severe', 'SLE_Moderate', 'COVID19_pro_Severe', 'HC_HC'])].copy()

In [127]:
# IFN-I gene expression across monocyte subclusters (Fig. S5C)

genes_ifn_i = ['ISG15', 'IFITM3', 'IFI44L', 'IFIT1', 'IFI6', 'MX1', 'SIGLEC1']

sc.pl.dotplot(
    adata_sub,
    var_names=genes_ifn_i,
    groupby='group',
    standard_scale='var',
    dot_max=1,
    dot_min=0,
    cmap='OrRd',
    show=False,
    save='IFN_I_genes_group.pdf',
)

{'mainplot_ax': <Axes: >,
 'size_legend_ax': <Axes: title={'center': 'Fraction of cells\nin group (%)'}>,
 'color_legend_ax': <Axes: title={'center': 'Mean expression\nin group'}>}

In [130]:
# score the IFN-I response module for monocytes
sc.tl.score_genes(adata_sub, gene_list=genes_ifn_i, score_name='IFN_I_response', use_raw=False)

computing score 'IFN_I_response'
    finished (0:00:03)


In [128]:
# Calculate CS score without IFN-I response genes

# Original CS score genes
cs_genes = [
    # Classical cytokines (both pro-inflammatory and regulatory cytokines)
    'IL1A', 'IL1B', 'IL6', 'TNF', 'IFNG', 'IL10', 'IL18', 'IFNB1', 'CSF2', 'CSF3',
    # Key receptors for key cytokines
    'IL1R1', 'IL6R', 'IL6ST', 'TNFRSF1A', 'TNFRSF1B','CSF2RA', 'CSF3R', 'IFNAR1', 'IFNAR2',
    # Key chemokines and their receptors
    'CCL2', 'CCR2', 'CCL5', 'CCR5', 'CXCR1', 'CXCR2', 'CXCL1',
    # Cytokine-induced/related signaling and regulatory molecules
    'NFKB1', 'NFKBIA', 'NFKBIZ', 'STAT1', 'STAT3', 'SOCS3', 'IRF4', 'JUN', 'FOS', 'IFI27',
    'TNFAIP6', 'CEBPB', 'PTGS2', 
    # Innate immune hyperactivation markers
    'S100A8', 'S100A9', 'S100A12', 'NLRP3', 'CASP1', 'GSDMD', 'TLR2',
    'ISG15', 'MX1'
]

# Remove genes that also belong to the IFN-I response module
ifn_genes = set(gene_dict['Type_I_IFN_response'])
cs_genes_no_ifn = [gene for gene in cs_genes if gene not in ifn_genes]

# Calculate CS_score_nifn 
sc.tl.score_genes(adata_sub, gene_list=cs_genes_no_ifn, score_name='CS_score_nifn', use_raw=False)

computing score 'CS_score_nifn'
    finished (0:00:03)


In [ ]:
# Loop over diseases and plot sample-level IFN-I vs CS_score_nifn in one figure

diseases = sorted(adata_sub.obs['disease'].dropna().unique())
fig, axes = plt.subplots(1, len(diseases), figsize=(4 * len(diseases), 4), squeeze=False)
axes = axes.flatten()

corr_results = []

for ax, disease in zip(axes, diseases):
    # Subset by disease
    adata_dis = adata_sub[adata_sub.obs['disease'] == disease].copy()

    # Use all monocytes together
    df_scores = adata_dis.obs[['sample_id', 'IFN_I_response', 'CS_score_nifn']].dropna().copy()

    # Modified Z-score outlier removal for both scores
    for score_col in ['IFN_I_response', 'CS_score_nifn']:
        x = df_scores[score_col]
        median = np.median(x)
        mad = np.median(np.abs(x - median))
        if mad == 0:
            mad = 1e-9
        modified_z = 0.6745 * (x - median) / mad
        df_scores = df_scores[np.abs(modified_z) <= 3.5]

    # Sample-level averages
    sample_avg_scores = df_scores.groupby('sample_id')[['IFN_I_response', 'CS_score_nifn']].mean().dropna()

    ifn_i = sample_avg_scores['IFN_I_response'].values
    cs_score = sample_avg_scores['CS_score_nifn'].values

    # Correlation + panel plot
    if len(sample_avg_scores) > 2 and np.std(ifn_i) > 0 and np.std(cs_score) > 0:
        corr, pval = pearsonr(ifn_i, cs_score)
        sns.regplot(
            data=sample_avg_scores,
            x='IFN_I_response',
            y='CS_score_nifn',
            robust=True,
            ax=ax,
            scatter_kws={'s': 50, 'alpha': 0.7},
            line_kws={'color': 'red'}
        )
        ax.set_title(f"{disease}\nR={corr:.2f}, p={pval:.2e}")
    else:
        corr, pval = np.nan, np.nan
        ax.text(0.5, 0.5, "Not enough samples", ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f"{disease}")

    ax.set_xlabel('Sample-average IFN-I response')
    ax.set_ylabel('Sample-average CS_score_nifn')

    corr_results.append({
        'disease': disease,
        'n_samples': len(sample_avg_scores),
        'pearson_r': corr,
        'p_value': pval
    })

plt.tight_layout()
plt.savefig("All_diseases_IFN_I_vs_CS_correlation_panels.pdf", dpi=300, bbox_inches='tight')
plt.show()

df_corr_results = pd.DataFrame(corr_results)
print(df_corr_results)

# 9. SLE-specific monocyte IFN/Fc receptor analyses
---
Fig. 5F

In [ ]:
# Subset to only SLE_Severe
adata_ssle = adata_m[adata_m.obs['group'] == 'SLE_Severe'].copy()

# Drop subclusters that are sample-biased
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']
adata_ssle = adata_ssle[~adata_ssle.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

# Define genes to be plotted for different monocyte subclusters in Severe SLE
marker_receptors = {
    'IFN_sensing': ['IFNAR1', 'IFNAR2'],
    'Fcy_receptor': ['FCGR1A', 'FCGR2A', 'FCGR3A'],
    'Signaling_adaptors': ['FCER1G', 'TYROBP']
}

# Plot dotplot
sc.pl.dotplot(
    adata_ssle,
    var_names=marker_receptors,
    groupby='myeloid_anno',
    standard_scale='var',
    dot_min = 0,
    dot_max = 1,
    cmap='RdBu_r',
    save='SLE_Severe_Monocyte_Marker_Dotplot.pdf'
)

# 10. CAR-T-associated monocyte receptor analyses

In [137]:
# Subset to CAR-T_CRS
adata_sub = adata_m[adata_m.obs['disease'] == 'CAR-T_CRS'].copy()

# Remove subclusters that are sample-biased or not relevant
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']
adata_sub = adata_sub[~adata_sub.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

In [ ]:
FCb_samples = [
        "CART_S001",
        "CART_S008",
        "CART_S015",
        "CART_S022",
        "CART_S029",
        "CART_S036",
        "CART_S043",
        "CART_S049",
        "CART_S056",
        "CART_S063",
        "CART_S070",
        "CART_S077",
        "CART_S084",
        "CART_S090",
        "CART_S097",
        "CART_S104"]

# Exclude FCb samples
adata_sub = adata_sub[~adata_sub.obs['sample_id'].isin(FCb_samples)].copy()

## Monocyte priming susceptibility
---
Fig. 3G, Fig. 3H

In [139]:
# Define the final CellChat-derived monocyte-receptor gene set.
gene_list = ['CD40', 'TNFRSF1A', 'TNFRSF1B', 'CSF2RA', 'CSF2RB', 'LTBR', 'ICAM1','ITGAL', 'ITGB2', 'ITGAM']

# Calculate the gene module score for 'monocyte_receptor'
valid_genes = [gene for gene in gene_list if gene in adata_sub.raw.var_names]
if len(valid_genes) > 0:
    sc.tl.score_genes(adata_sub, gene_list=valid_genes, score_name='monocyte_receptor_score', use_raw=False)
else:
    print("Warning: No valid genes found for monocyte_receptor_score")

computing score 'monocyte_receptor_score'
    finished (0:00:01)


In [ ]:
# Fig. 3G

from itertools import combinations

# subset to CAR-T_CRS_before
adata_before = adata_sub[adata_sub.obs['stage'] == 'CAR-T_CRS_before'].copy()

# Sample-level means per severity (collapse to CRS vs No_CRS)
df_scores = adata_before.obs[['sample_id', 'severity', 'monocyte_receptor_score']].dropna().copy()
df_scores['severity'] = df_scores['severity'].cat.add_categories(['CRS'])
df_scores['severity'] = df_scores['severity'].where(df_scores['severity'] == 'No_CRS', 'CRS')

sample_means = (
    df_scores.groupby(['sample_id', 'severity'], observed=True)['monocyte_receptor_score']
    .mean()
    .reset_index()
)

severity_order = ['No_CRS', 'CRS']

# Boxplot
plt.figure(figsize=(6, 4))
sns.boxplot(
    data=sample_means,
    x='severity',
    y='monocyte_receptor_score',
    order=severity_order,
    showfliers=False,
    width=0.6
)
sns.stripplot(
    data=sample_means,
    x='severity',
    y='monocyte_receptor_score',
    order=severity_order,
    color='black',
    size=4,
    alpha=0.6,
    jitter=0.2
)
plt.xlabel('Severity')
plt.ylabel('Sample-mean monocyte_receptor_score')
plt.tight_layout()
plt.savefig('CAR-T_CRS_before_monocyte_receptor_score.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Output the sample means to CSV for record-keeping
sample_means.to_csv('CAR-T_CRS_before_sample_means_monocyte_receptor_score.csv', index=False)

# Pairwise tests with BH correction
results = []
pairs = list(combinations(severity_order, 2))

pvals = []
for a, b in pairs:
    x = sample_means.loc[sample_means['severity'] == a, 'monocyte_receptor_score']
    y = sample_means.loc[sample_means['severity'] == b, 'monocyte_receptor_score']
    if len(x) > 0 and len(y) > 0:
        stat, p = stats.mannwhitneyu(x, y, alternative='two-sided')
    else:
        stat, p = np.nan, np.nan
    results.append({'group_a': a, 'group_b': b, 'stat': stat, 'p_value': p})
    pvals.append(p)

# Benjamini-Hochberg adjustment
pvals = np.array(pvals, dtype=float)
mask = ~np.isnan(pvals)
m = mask.sum()
adj = np.full_like(pvals, np.nan)

if m > 0:
    order_idx = np.argsort(pvals[mask])
    ranked = pvals[mask][order_idx] * m / (np.arange(1, m + 1))
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adj_vals = np.empty_like(ranked)
    adj_vals[np.arange(m)] = ranked
    adj[mask] = adj_vals[np.argsort(order_idx)]

# Collect results
df_pairwise = pd.DataFrame(results)
df_pairwise['p_adj_bh'] = adj
print(df_pairwise)


In [ ]:
# Fig. 3G
# Dotplot of gene expressions of genes in gene_list across CRS and No_CRS groups in CAR-T_CRS samples
sc.pl.dotplot(
    adata_before,
    var_names=gene_list,
    groupby='severity',
    standard_scale='var',
    dot_min=0,
    dot_max=1,
    cmap='RdBu_r',
    save='CAR-T_CRS_before_monocyte_receptor_genes_dotplot.pdf'
)

## Per-subcluster CS score across CRS stages
---
Fig. S3I

In [ ]:
# Subset to CAR-T_CRS
adata_sub = adata_m[adata_m.obs['disease'] == 'CAR-T_CRS'].copy()

# Remove subclusters that are sample-biased or not relevant
clusters_to_remove = ['Mono_CD14_IFI44', 'Mono_CD14_CD16', 'cDC_CD1C']
adata_sub = adata_sub[~adata_sub.obs['myeloid_anno'].isin(clusters_to_remove)].copy()

FCb_samples = [
        "CART_S001",
        "CART_S008",
        "CART_S015",
        "CART_S022",
        "CART_S029",
        "CART_S036",
        "CART_S043",
        "CART_S049",
        "CART_S056",
        "CART_S063",
        "CART_S070",
        "CART_S077",
        "CART_S084",
        "CART_S090",
        "CART_S097",
        "CART_S104"]

# Exclude FCb samples
adata_sub = adata_sub[~adata_sub.obs['sample_id'].isin(FCb_samples)].copy()

In [ ]:
# Ridge/density plot of globally scaled CS scores across CAR-T CRS stages by monocyte subcluster

# Extract and clean
df_cs = adata_sub.obs[['myeloid_anno', 'stage', 'CS_score']].dropna().copy()

subcluster_order = ['Mono_CD14_IL1B', 'Mono_CD14_S100A8', 'Mono_CD16_LST1']
stage_order = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']

df_cs = df_cs[
    df_cs['myeloid_anno'].isin(subcluster_order) &
    df_cs['stage'].isin(stage_order)
].copy()

# Global scaling
global_mean = df_cs['CS_score'].mean()
global_std = df_cs['CS_score'].std()
df_cs['CS_score_global_z'] = (df_cs['CS_score'] - global_mean) / global_std

# Summary csv (unfiltered, for reporting)
summary = (
    df_cs.groupby(['myeloid_anno', 'stage'], observed=True)
    .agg(
        n_cells=('CS_score', 'size'),
        mean_CS_score=('CS_score', 'mean'),
        median_CS_score=('CS_score', 'median'),
        mean_CS_score_global_z=('CS_score_global_z', 'mean'),
        median_CS_score_global_z=('CS_score_global_z', 'median')
    )
    .reset_index()
)
summary.to_csv('monocyte_CS_score_ridgeplot_summary.csv', index=False)

# MAD-based filtering for visualization only
def mad_filter(group):
    x = group['CS_score_global_z'].values
    median = np.median(x)
    mad = np.median(np.abs(x - median))
    if mad == 0:
        mad = 1e-9
    modified_z = 0.6745 * (x - median) / mad
    return np.abs(modified_z) <= 3.5

mask = (
    df_cs.groupby(['myeloid_anno', 'stage'], observed=True)
    .apply(mad_filter)
    .reset_index(level=[0, 1], drop=True)
)
mask = mask.explode().astype(bool).values
df_plot = df_cs.loc[mask].copy()

# plotting parameters
stage_colors = {
    'CAR-T_CRS_before': 'steelblue',
    'CAR-T_CRS_pro': 'firebrick',
    'CAR-T_CRS_con': 'green'
}

# x-axis limits shared across panels
# x_min = df_plot['CS_score_global_z'].min()
# x_max = df_plot['CS_score_global_z'].max()
xlim = (-4, 4)

fig, axes = plt.subplots(
    nrows=len(subcluster_order), ncols=1, figsize=(7, 8),
    sharex=True, constrained_layout=True
)

if len(subcluster_order) == 1:
    axes = [axes]

for ax, subcluster in zip(axes, subcluster_order):
    sub_df = df_plot[df_plot['myeloid_anno'] == subcluster]

    for stage in stage_order:
        vals = sub_df.loc[sub_df['stage'] == stage, 'CS_score_global_z']
        if len(vals) > 1:
            sns.kdeplot(
                vals,
                ax=ax,
                fill=True,
                alpha=0.3,
                linewidth=1.5,
                color=stage_colors[stage],
                bw_adjust=1.5,
                clip=xlim
            )

    ax.set_ylabel(subcluster, fontsize=11)
    ax.set_yticks([])
    ax.set_xlim(xlim)
    ax.grid(False)

# Legend
legend_handles = [
    plt.Line2D([0], [0], color=stage_colors[s], lw=4, label=s)
    for s in stage_order
]
axes[0].legend(handles=legend_handles, loc='upper right', frameon=False)

axes[-1].set_xlabel('Globally scaled CS score', fontsize=12)

sns.despine(left=True, bottom=False)
plt.savefig('monocyte_CS_score_ridgeplot.pdf', dpi=300, bbox_inches='tight')
plt.savefig('monocyte_CS_score_ridgeplot.png', dpi=300, bbox_inches='tight')
plt.show()